# A Federated Learning and Cryptographic Audit Trail Framework for Bangla Social Media Forensics and User Profiling

<br/>

**Authors:** Ashif Rabbani, Md. Sakib Muhtadee, Jannatul Ferdous

<br/>

This notebook contains the complete pipeline of the thesis:

* a **text normalisation** front end that recovers words hidden with `*` and converts Romanized Bangla to Bangla script,
* a **BanglaBERT classifier** for five classes of harmful content, trained with **simulated federated learning**,
* a **cryptographic hash-linked evidence ledger** (audit trail) that stores model updates, evidence and user profiles
  in tamper-evident form, and
* the **digital-forensic (DF) profiling** of public social-media accounts from their activity time series.

---
## 1. Problem statement: DF analysis of social-media user data

The investigator studies a **public Facebook user account**:

```
Facebook  ->  user account (public profile)
                 |
                 |-- About: public personal information (visibility, joined, followers ...)
                 |
                 |-- 1. Monitoring activities
                 |        -> status   (the user writes a post)
                 |        -> share    (the user shares another post)
                 |        -> comment  (the user comments on other posts)
                 |        =  TIME-SERIES DATA (every activity has a date/time)
                 v
             PROFILING with aspects 1..5  ->  profile vector  ->  RESULT: activity profile + risk level
```

The **5 profiling aspects** (Section 8):

| # | Aspect | Question it answers |
|---|--------|---------------------|
| 1 | Activity level | How active is the account and when? |
| 2 | Harmful ratio | What share of the activities is harmful? |
| 3 | Harm type & severity | What kind of harm (offensive, hate, violence, ...) and how serious? |
| 4 | Time trend | Is the harmful behaviour increasing (escalating) over time? |
| 5 | Behaviour channel | Does the user *write* (status), *spread* (share) or *attack others* (comment)? |

The aspects are combined into a **profile vector** and a **risk level** (Low / Medium / High).

---
## 2. Novel contribution: hidden and Romanized harmful comments

On social media, users hide abusive words with `*` and/or write Bangla in Latin letters (Romanized Bangla).
Standard classifiers miss these comments. The proposed pipeline first restores them to standard Bangla, then
classifies them, and records harmful ones as **tamper-evident evidence** in the hash-linked ledger:

```
Example 1 (Romanized + hidden letters)         Example 2 (Bangla + hidden letters)

   kutt*r bacc*                                    কুত্*ার বাচ্চা
        |  Step 1: Unmask (guess hidden letters)        |  Step 1: Unmask
        v                                               v
   kuttar baccha                                   কুত্তার বাচ্চা
        |  Step 2: Transliterate (Latin -> Bangla)
        v                                               |
   কুত্তার বাচ্চা                                         |
        |  Step 3: Classify                             |  Step 3: Classify
        v                                               v
   harmful class                                   harmful class
        |  Step 4: harmful -> record                    |  Step 4: record
        v                                               v
   LEDGER (tamper-evident evidence)                LEDGER
```

**Two problems are addressed:**
1. **Class imbalance:** the available data contains far more `normal` comments than harmful ones, so a model is
   biased towards predicting `normal`.
2. **Per-class F1:** the F1 score of every class is reported and compared, not only the overall accuracy.

**Approach:** a combined and cleaned dataset with a balanced training core (Section 1), over-sampling of the rare
harmful classes inside every federated client (Section 4), and per-class decision thresholds tuned on the validation
set (Section 6). The test set keeps the natural class distribution, and the improved model is compared with a
baseline class by class, over three seeds (Section 10).

---
## 3. How the parts work together

Every activity of a monitored account (Section 8) passes through the novel pipeline
(**unmask → transliterate → classify**) before profiling. Harmful activities become `EVIDENCE` blocks and the final
profile becomes a `USER_PROFILE` block, so a later change to the investigation record is detected (Section 9).

## 4. Notebook structure

| Section | Content | Part |
|---|---|---|
| 0 | Setup (libraries, configuration) | – |
| 1 | Datasets (+ held-out BanTH pairs for evaluation) | – |
| 2 | Text normalisation: clean → unmask → transliterate, **quantitative evaluation (2.6)** | Novel |
| 3 | Cryptographic hash-linked evidence ledger | Novel |
| 4 | Class imbalance (Problem 1), IID and Non-IID client splits | Novel |
| 5 | Simulated federated learning classifier + centralized reference | – |
| 6 | Results: per-class F1, baseline vs improved (Problem 2) | Novel |
| 7 | End-to-end analysis of a comment | Novel |
| 8 | DF analysis of user accounts: activities, time series, profiling, **sensitivity analysis (8.6)** | Requirement |
| 9 | Ledger verification and **tamper experiments**, saved outputs, requirement checklist | – |
| 10 | **Robustness experiments**: seeds 42–44, bootstrap, centralized vs IID vs Non-IID, human reference labels | Revision |

## 5. Execution

`RUN_MODE` in cell 0.2 selects the configuration: `FULL` (all training data, 10 federated rounds, BanglaT5
fine-tuning) produces the reported results; `QUICK` (6,000 training comments, 3 rounds, about 15 minutes on a T4)
is a short functional check. Run the notebook once per dataset version (`DATA_VERSION` in 1.1: A, B, C);
Section 10 then adds the extra seeds and, for version C, the federated-learning study.
Data access is described in `README.md`.

<a href="https://colab.research.google.com/github/usamaalmahi/Bangla-Social-Media-Digital-Forensics/blob/main/Bangla_Forensics_FL_UserProfiling_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
# Section 0 · Setup

In [ ]:
# ============================================================
# 0.1  Install libraries (most are already installed in Colab)
# ============================================================
!pip install -q transformers sentencepiece datasets kagglehub scikit-learn seaborn openpyxl

In [ ]:
# ============================================================
# 0.2  Imports and main settings
# ============================================================
import os, re, json, copy, glob, html, time, random, hashlib, warnings
from io import BytesIO
from collections import Counter, defaultdict
from datetime import datetime, timezone, timedelta

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM
from transformers.utils import logging as hf_logging
from sklearn.model_selection import train_test_split
from sklearn.metrics import (f1_score, accuracy_score, precision_score, recall_score, classification_report,
                             confusion_matrix, cohen_kappa_score)

hf_logging.set_verbosity_error()        # hide long HuggingFace info messages
warnings.filterwarnings("ignore")

# ---------------- Main switches ----------------
RUN_MODE = "FULL"        # @param ["QUICK", "FULL"]
# QUICK = 6,000 training comments, 3 rounds           -> about 15 min on a T4 GPU (functional check)
# FULL  = all training comments, 10 rounds + BanglaT5 -> reported results

SAVE_TO_DRIVE = True    # @param {type:"boolean"}
# True = save all results in Google Drive, so they are not lost when the Colab session ends

# ---------------- Fixed settings ----------------
SEED = 42               # @param {type:"integer"}   main seed; Section 10 repeats training with EXTRA_SEEDS

def set_seed(seed):
    """The same random numbers in Python, NumPy and PyTorch (CPU and GPU) for a given seed."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(SEED)
DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"               # mixed precision: about 2x faster training on a GPU

# The 5 classes. The position in this list is the class number the model uses.
LABELS   = ["normal", "offensive", "cyberbully", "hate_speech", "violence"]
LABEL2ID = {name: i for i, name in enumerate(LABELS)}
CLASS_COLORS = {"normal": "#4C9F70", "offensive": "#E1B12C", "cyberbully": "#8E44AD",
                "hate_speech": "#E67E22", "violence": "#C0392B"}

CLASSIFIER_MODEL = "csebuetnlp/banglabert"   # Bangla BERT model used by every FL client
T5_MODEL         = "csebuetnlp/banglat5"     # Bangla T5 model used for transliteration
NUM_CLIENTS      = 3                         # number of simulated federated clients (e.g. 3 organisations)
LEARNING_RATE    = 3e-5

# A harmful prediction is recorded as ledger EVIDENCE only when the model is at least
# this sure. Less certain predictions are marked "uncertain" for a human to review.
EVIDENCE_MIN_CONFIDENCE = 0.60

# ---------------- Robustness experiments (Section 10, run after the main pipeline) ----------------
EXTRA_SEEDS         = [43, 44]                # baseline + improved are repeated with these seeds (mean ± std)
RUN_EXTRA_SEEDS     = True                    # @param {type:"boolean"}
RUN_FL_STUDY        = True                    # @param {type:"boolean"}  centralized + Non-IID runs (version C)
FL_STUDY_PARTITIONS = ["dirichlet", "source"] # Non-IID client splits compared with the IID split (4.2)
DIRICHLET_ALPHA     = 0.5                     # smaller alpha = more different class mixes between clients

# ---------------- Settings that depend on RUN_MODE ----------------
if RUN_MODE == "QUICK":
    MAX_ROWS       = 6000    # use only a sample of the TRAINING data (class proportions are kept)
    FL_ROUNDS      = 3       # federated rounds
    LOCAL_EPOCHS   = 2       # epochs each client trains per round
    MAX_LEN        = 64      # max tokens per comment
    BATCH_SIZE     = 16
    TRAIN_BANGLAT5 = False   # BanglaT5 training is slow, so it is skipped in QUICK mode
else:
    MAX_ROWS       = None    # use all training rows of the chosen dataset version
    FL_ROUNDS      = 10      # the best round is kept (5.2), so extra rounds cannot make the kept model worse
    LOCAL_EPOCHS   = 1
    MAX_LEN        = 128
    BATCH_SIZE     = 16
    TRAIN_BANGLAT5 = True

# ---------------- Output folder (figures, models, ledger, profiles) ----------------
OUT_DIR = "outputs"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/bangla_forensics_outputs"
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

sns.set_theme(style="whitegrid")
def save_fig(fig, name):
    """Save a figure as a 300-dpi PNG (print quality, for the report)."""
    path = os.path.join(FIG_DIR, name + ".png")
    fig.savefig(path, dpi=300, bbox_inches="tight")
    print("Figure saved:", path)

print("Run mode :", RUN_MODE)
print("Device   :", DEVICE, "(mixed precision on)" if USE_AMP else "")
print("Outputs  :", OUT_DIR)
if DEVICE == "cpu":
    print("WARNING: no GPU found. Training will be very slow. Use Runtime -> Change runtime type -> T4 GPU.")

---
# Section 1 · Load the datasets

**Forensic dataset (5 classes).** Seven public Bangla datasets are combined into one file by
`dataset/build_dataset.py` (in the project repository). The script maps every source label to our 5 classes with one
written guideline (`dataset/guideline_v1.md`), removes duplicates, drops comments that two sources label differently,
fixes the train / validation / test split, and flags probable label errors in the training part.

| Dataset | Platform | Rows | Main use |
|---|---|---|---|
| **BanHate** | YouTube | 19,203 | all classes |
| **BD-SHS** (train + val + test files) | social media | 50,281 | cyberbully, violence, normal |
| **Bangla Online Comments** (Mendeley, CC BY 4.0) | Facebook | 44,001 | cyberbully, hate_speech |
| **Belal et al. toxic comments** | social media | 16,073 | cyberbully, hate_speech, offensive |
| **BLP-2023 VITD** | YouTube | 6,046 | violence |
| **TB-OLID** (Romanized Bangla, training only) | Facebook | 5,000 | offensive, cyberbully, normal |
| **Bangla vulgar corpus** (drama reviews, training only) | YouTube | 664 | offensive |

TB-OLID and the vulgar corpus give the small `offensive` class more real examples. They are used for
**training only**, so the validation and test sets contain only the five original sources.

Three **dataset versions** can be trained (`DATA_VERSION` in 1.1). Validation and test are the same for all three:

| Version | Training data |
|---|---|
| **A** | BanHate + BD-SHS only (the 2 original sources) |
| **B** | all sources |
| **C** | all sources, probable label errors removed, balanced core: the same number of comments per class (recommended) |

**Data access.** The combined file `data/bangla_forensics_dataset_v1.csv.gz` (about 8 MB) is kept in the
**private** project repository, because two sources have no licence that allows redistribution. Cell 1.1
downloads it with a read-only GitHub token stored as the Colab secret `GITHUB_TOKEN` (the token is never
written into the notebook); the file can also be uploaded to the Colab session directly. Full provenance,
licences and processing steps are documented in `dataset/DATASHEET.md`.

**Other resources:**

| Dataset | Language | Used for |
|---|---|---|
| **BanTH** | Romanized Bangla ↔ Bangla | transliteration (English letters → Bangla) |
| **Nirmol + ToxLex** | Bangla profane words and phrases | unmasking hidden bad words |


In [ ]:
# ============================================================
# 1.1  Forensic dataset: 7 public sources, combined and cleaned
# ============================================================
# Built once by dataset/build_dataset.py and stored (compressed) in the PRIVATE GitHub repository.
# Colab reads it with the read-only token saved in Colab Secrets as GITHUB_TOKEN (see README.md).
DATA_VERSION = "C"       # @param ["A", "B", "C"]
EQUAL_REAL_ONLY = False  # @param {type:"boolean"}
# Version C only. False = 6,000 per class; the smaller 'offensive' class is filled up by repeating comments (4.2).
#                True  = every class cut down to the size of the smallest one (3,701), real comments only.
# A = BanHate + BD-SHS only, all training rows      (the 2 original sources)
# B = all sources, all training rows
# C = all sources, probable label errors removed, balanced core (same size per class)   (recommended)

GITHUB_REPO   = "usamaalmahi/Bangla-Social-Media-Digital-Forensics"
GITHUB_BRANCH = "main"
DATASET_PATH  = "data/bangla_forensics_dataset_v1.csv.gz"     # path inside the repository
LOCAL_COPY    = "bangla_forensics_dataset_v1.csv.gz"          # downloaded once per Colab session

def github_token():
    """The read-only GitHub token from Colab Secrets (never type the token itself into the notebook)."""
    try:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception as e:
        print(f"GITHUB_TOKEN secret not available ({type(e).__name__}).")
        return None

def download_from_github(path, dest):
    """Download one file from the private repository through the GitHub API."""
    #token = github_token()
    #if not token:
    #    raise RuntimeError("Add the GITHUB_TOKEN secret (key icon on the left, 'Notebook access' on), or upload "
    #                       f"'{LOCAL_COPY}' to Colab by hand, then run this cell again.")
    response = requests.get(
        f"https://api.github.com/repos/{GITHUB_REPO}/contents/{path}?ref={GITHUB_BRANCH}", timeout=300,
        headers={"Accept": "application/vnd.github.raw+json",
                 "X-GitHub-Api-Version": "2022-11-28"})
    if response.status_code in (401, 403, 404):
        raise RuntimeError(f"GitHub answered {response.status_code} for {path}. Check that the token is valid, has "
                           "'Contents: Read-only' for this repository, and that the file is pushed to "
                           f"the '{GITHUB_BRANCH}' branch.")
    response.raise_for_status()
    with open(dest, "wb") as f:
        f.write(response.content)
    print(f"Downloaded {path} ({len(response.content) / 1e6:.1f} MB)")

if not os.path.isfile(LOCAL_COPY):
    download_from_github(DATASET_PATH, LOCAL_COPY)

full_dataset = pd.read_csv(LOCAL_COPY, encoding="utf-8-sig")    # pandas unpacks the .gz by itself
full_dataset = full_dataset[full_dataset["status"] == "kept"]
in_train = full_dataset["split"] == "train"
if DATA_VERSION == "A":
    in_train &= full_dataset["source"].isin(["banhate", "bdshs"])
elif DATA_VERSION == "C":
    in_train &= full_dataset["train_core"] & ~full_dataset["train_filtered_out"]
if DATA_VERSION == "C" and EQUAL_REAL_ONLY:
    core = full_dataset[in_train]
    smallest = core["label"].value_counts().min()
    keep = core.groupby("label", group_keys=False).apply(lambda g: g.sample(smallest, random_state=SEED)).index
    in_train = full_dataset.index.isin(keep)
    print(f"EQUAL_REAL_ONLY: {smallest} real comments per class, no repeated comments")
    DATA_VERSION = "C_equal"     # saved and compared (6.5, 6.6) as its own version

# Outputs of this dataset version (figures, ledgers, tables, models) go to their own folder, so running another
# version does not overwrite them. Run files (runs_v2/) and BanglaT5 stay in OUT_DIR and are shared by all versions.
RUN_DIR = os.path.join(OUT_DIR, f"version_{DATA_VERSION}")
FIG_DIR = os.path.join(RUN_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)
print("Outputs of this version:", RUN_DIR)
data = full_dataset[in_train | full_dataset["split"].isin(["val", "test"])]

# QUICK mode: a sample of the training rows (same class proportions); validation and test stay complete
train_part = data[data["split"] == "train"]
if MAX_ROWS and len(train_part) > MAX_ROWS:
    train_part, _ = train_test_split(train_part, train_size=MAX_ROWS, stratify=train_part["label"],
                                     random_state=SEED)
data = pd.concat([train_part, data[data["split"] != "train"]]).reset_index(drop=True)

print(f"Dataset version {DATA_VERSION}: train {(data['split'] == 'train').sum()} | "
      f"validation {(data['split'] == 'val').sum()} | test {(data['split'] == 'test').sum()}")
print(pd.crosstab(data["source"], data["split"]))
data[["text", "source", "label", "split"]].head()


In [ ]:
# ============================================================
# 1.2  Class distribution  (this shows Problem 1)
# ============================================================
counts = data.loc[data["split"] == "train", "label"].value_counts().reindex(LABELS, fill_value=0)
print(pd.DataFrame({"count": counts, "percent": (100 * counts / counts.sum()).round(1)}))
print(f"\nImbalance: 'normal' has {counts['normal'] / max(counts.drop('normal').min(), 1):.1f}x "
      f"more comments than the smallest class.")

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(LABELS, counts.values, color=[CLASS_COLORS[c] for c in LABELS])
for i, v in enumerate(counts.values):
    ax.text(i, v, f"{v}\n({100 * v / counts.sum():.1f}%)", ha="center", va="bottom", fontsize=9)
ax.set_ylim(0, counts.max() * 1.2)
ax.set_title(f"Problem 1: class distribution of the training data (version {DATA_VERSION})")
ax.set_ylabel("Training comments")
save_fig(fig, "1_class_distribution")
plt.show()

In [ ]:
# ============================================================
# 1.3  Transliteration dataset: BanTH  (Romanized Bangla <-> Bangla)
# ============================================================
def load_banth():
    from datasets import load_dataset
    ds = load_dataset("aplycaebous/BanTH")
    df = pd.concat([ds[split].to_pandas() for split in ds.keys()], ignore_index=True)
    df.columns = [c.lower() for c in df.columns]   # columns used: text (Romanized), bangla
    df = df[["text", "bangla"]].dropna().rename(columns={"text": "roman"})
    return df.astype(str).reset_index(drop=True)

try:
    banth = load_banth()
    print("BanTH:", len(banth), "sentence pairs loaded")
except Exception as e:
    print(f"BanTH download failed ({e}) - using a tiny built-in sample.")
    banth = pd.DataFrame([
        ("Ami tomake bhalobasi", "আমি তোমাকে ভালোবাসি"),
        ("Kemon acho",           "কেমন আছো"),
        ("Valo achi dhonnobad",  "ভালো আছি ধন্যবাদ"),
        ("Tumi kothay",          "তুমি কোথায়"),
        ("Bangla amar bhasha",   "বাংলা আমার ভাষা"),
    ], columns=["roman", "bangla"])


# A fixed set of BanTH pairs is HELD OUT. It is never used for the dictionary (2.3), the unmasker vocabulary (2.2)
# or BanglaT5 (2.4), so the normalisation is evaluated on sentences the pipeline has never seen (2.6).
# The held-out set does not depend on SEED, so it is identical in every run.
NORM_EVAL_PAIRS = 300
n_words = banth["roman"].str.split().str.len()
candidates = banth[n_words.between(3, 15) & (banth["roman"].str.len() < 200)]
banth_heldout = candidates.sample(min(NORM_EVAL_PAIRS, len(candidates) // 10), random_state=7).reset_index(drop=True)
banth = banth[~banth["roman"].isin(banth_heldout["roman"]) &
              ~banth["bangla"].isin(banth_heldout["bangla"])].reset_index(drop=True)
print(f"BanTH: {len(banth)} pairs for the dictionary and BanglaT5 | {len(banth_heldout)} pairs held out (2.6)")
banth.head()

In [ ]:
# ============================================================
# 1.4  Profane word and phrase lists: Nirmol + ToxLex
# ============================================================
# These lists help the unmasker: when a word is hidden with '*', it is most
# likely a bad word, so bad words and bad phrases are preferred as answers.

TOXLEX_URL = ("https://data.mendeley.com/public-files/datasets/9pz8ssmc49/files/"
              "c13b54f5-30d7-4357-8c49-37bc46e510ab/file_downloaded")
TOXLEX_REPO_PATH = "data/ToxLex.xlsx"   # fallback: a copy in the private project repository (Mendeley often blocks Colab)
TOXLEX_FILE = "ToxLex.xlsx"            # local copy; the file can also be uploaded to Colab under this name

def load_nirmol():
    """Nirmol (GitHub): one Bangla slang word or phrase per line."""
    if not os.path.isdir("Nirmol"):
        os.system("git clone -q --depth 1 https://github.com/Sigmakib2/Nirmol.git")
    df = pd.read_csv("Nirmol/datasets/Nirmol-v1-dataset.csv", header=None)   # the file has NO header row
    return df[0].dropna().astype(str).tolist()

def load_toxlex():
    """ToxLex (Mendeley Data): Excel file with 1,959 toxic Bangla bigrams."""
    errors = []
    if not os.path.isfile(TOXLEX_FILE):
        try:
            # A browser-like User-Agent: some servers block the default Python one
            response = requests.get(TOXLEX_URL, timeout=60,
                                    headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"})
            response.raise_for_status()
            if not response.content.startswith(b"PK"):       # every .xlsx file starts with "PK"
                raise ValueError("the server did not send the Excel file (it may block Colab)")
            with open(TOXLEX_FILE, "wb") as f:
                f.write(response.content)
        except Exception as e:
            errors.append(f"Mendeley: {type(e).__name__}: {e}")
    if not os.path.isfile(TOXLEX_FILE):
        try:
            download_from_github(TOXLEX_REPO_PATH, TOXLEX_FILE)   # same read-only token as cell 1.1
        except Exception as e:
            errors.append(f"repository: {type(e).__name__}: {e}")
    if not os.path.isfile(TOXLEX_FILE):
        raise RuntimeError(" | ".join(errors))
    df = pd.read_excel(TOXLEX_FILE)
    column = "Base_bigram" if "Base_bigram" in df.columns else df.columns[0]
    entries = df[column].dropna().astype(str).tolist()
    if not entries:
        raise ValueError("the file has no entries")
    return entries

profane_entries = []
for name, loader in [("Nirmol", load_nirmol), ("ToxLex", load_toxlex)]:
    try:
        entries = loader()
        profane_entries += entries
        print(f"{name}: {len(entries)} entries")
    except Exception as e:
        print(f"{name}: download failed ({type(e).__name__}: {e})")
        if name == "ToxLex":
            print(f"   -> continuing without ToxLex (it can be uploaded to Colab as '{TOXLEX_FILE}')")

# A few common words written in both scripts. They make sure the examples from
# the novel part always work, even when a download fails.
SEED_ROMAN_TO_BANGLA = {
    "kutta": "কুত্তা", "kuttar": "কুত্তার", "bacca": "বাচ্চা", "baccha": "বাচ্চা",
    "shala": "শালা", "gadha": "গাধা", "shuor": "শুয়োর", "shuorer": "শুয়োরের", "harami": "হারামি",
}
print("Profane entries loaded:", len(profane_entries))

---
# Section 2 · Novel part – text normalisation: clean → unmask → transliterate

**Step 0 – Clean:** remove links, emojis, repeated punctuation and extra spaces.

**Step 1 – Unmask:** a hidden word like `kutt*r` becomes a search pattern `kutt???r`
(each `*` = 1 to 3 unknown letters). We look up all known words that fit the pattern:
* **Phrase first:** the hidden word **and its neighbour** are compared with the bad **phrases** from Nirmol/ToxLex
  (e.g. `কু*ার বা*া` fits the phrase `কুত্তার বাচ্চা`). Using the neighbour gives the right answer more often.
* **Then single words:** a known bad word is preferred, then the most frequent word.
* Words written as `k-u-t-t-a` are simply joined.

This is simple, fast, needs no training and works for **both** Bangla and Romanized text.

**Step 2 – Transliterate:** Romanized words are replaced using a **word dictionary** learned automatically from
BanTH sentence pairs. Different spellings of the same word (`bhalobasi`, `valobashi`, `bhalobashi`) are matched with a
**spelling key**. If a sentence still has unknown words, a fine-tuned **BanglaT5** model translates the whole
sentence (FULL mode only).

In [ ]:
# ============================================================
# 2.1  Basic cleaning + small text helpers
# ============================================================
EMOJI_RE = re.compile("[\U0001F300-\U0001FAFF\U00002600-\U000027BF\U0001F1E0-\U0001F1FF]+")

def clean_text(text):
    """Remove noise that carries no meaning for classification."""
    if not isinstance(text, str):
        return ""
    text = html.unescape(text)                          # &amp; -> &
    text = re.sub(r"http\S+|www\.\S+", " ", text)       # links
    text = EMOJI_RE.sub(" ", text)                      # emojis
    text = text.replace("​", "").replace("﻿", "")   # invisible characters
    text = re.sub(r"[@#](\w+)", r"\1", text)            # @ name / #tag -> name / tag
    text = re.sub(r"([!?।.])\1+", r"\1", text)          # !!! -> !
    return re.sub(r"\s+", " ", text).strip()            # extra spaces

def split_words(text):
    """Split text into lower-case words without punctuation."""
    return re.findall(r"[^\s!?,;:।.\"'()\[\]{}]+", str(text).lower())

def split_punctuation(token):
    """'"kutt*r!' -> ('"', 'kutt*r', '!')  so punctuation does not disturb the lookup."""
    return re.match(r"^([\"'(\[]*)(.*?)([!?,;:।.\"')\]]*)$", token, re.S).groups()

def has_bangla(text): return bool(re.search(r"[ঀ-৿]", text))   # Bangla Unicode block
def has_roman(text):  return bool(re.search(r"[a-zA-Z]", text))

print(clean_text("দারুণ ভিডিও!!! 😂😂 https://fb.com/xyz   #Dhaka"))

In [ ]:
# ============================================================
# 2.2  Step 1 - Unmasker (phrase + word pattern matching)
# ============================================================

# Profane entries are split into single BAD WORDS and multi-word BAD PHRASES.
# (Splitting a phrase like 'কুত্তার বাচ্চা' into words would wrongly mark
#  harmless words such as 'ছেলে' (boy) as bad words.)
profane = {" ".join(split_words(clean_text(e))) for e in profane_entries} - {""}
BAD_WORDS   = {p for p in profane if " " not in p}
BAD_WORDS  |= set(SEED_ROMAN_TO_BANGLA) | set(SEED_ROMAN_TO_BANGLA.values())
BAD_PHRASES = {p for p in profane if " " in p}
BAD_PHRASES |= {f"{a} {b}" for a in ["kuttar", "shuorer"] for b in ["bacca", "baccha"]}   # Romanized

# Vocabulary = every word we have seen, with how often we saw it.
VOCAB = Counter()
for text in list(data["text"]) + list(banth["roman"]) + list(banth["bangla"]):
    VOCAB.update(w for w in split_words(clean_text(text)) if "*" not in w)
VOCAB.update(w for p in BAD_WORDS | BAD_PHRASES for w in p.split())   # every bad word is known

# Group words by length so the search is fast
VOCAB_BY_LENGTH = defaultdict(list)
for word in VOCAB:
    VOCAB_BY_LENGTH[len(word)].append(word)


def mask_to_pattern(word):
    """'kutt*r' -> regex 'kutt.{1,3}r' (each group of '*' = 1 to 3 hidden characters)."""
    pieces = re.split(r"\*+", word.lower())
    return ".{1,3}".join(re.escape(p) for p in pieces), "".join(pieces), len(pieces) - 1


def unmask_word(word):
    """Guess the hidden letters of ONE word, e.g. 'kutt*r' -> 'kuttar'."""
    if re.fullmatch(r"(\S[-/])+\S", word):       # k-u-t-t-a  ->  kutta
        return re.sub(r"[-/]", "", word)
    if "*" not in word:
        return word
    pattern, known, stars = mask_to_pattern(word)
    if len(known) < 2:                           # too little information to guess
        return word
    regex = re.compile("^" + pattern + "$")
    candidates = [w for length in range(len(known) + stars, len(known) + 3 * stars + 1)
                  for w in VOCAB_BY_LENGTH.get(length, []) if regex.match(w)]
    if not candidates:
        return word
    # Prefer a known bad word (people usually hide bad words), then the most frequent word
    return max(candidates, key=lambda w: (w in BAD_WORDS, VOCAB[w]))


def unmask_phrase(first, second):
    """Two neighbouring words (at least one hidden) -> the matching bad phrase, or None."""
    parts = [mask_to_pattern(w)[0] if "*" in w else re.escape(w.lower()) for w in (first, second)]
    regex = re.compile("^" + " ".join(parts) + "$")
    candidates = [p for p in BAD_PHRASES if regex.match(p)]
    if not candidates:
        return None
    return max(candidates, key=lambda p: sum(VOCAB[w] for w in p.split())).split()


def unmask_text(text):
    """Unmask a sentence: first try word pairs against bad phrases, then single words."""
    tokens = [split_punctuation(t) for t in text.split()]     # (head, word, tail) for each token
    words = [core for _, core, _ in tokens]
    fixed = {}                                                # position -> unmasked word
    for i in range(len(words) - 1):
        if i not in fixed and ("*" in words[i] or "*" in words[i + 1]):
            match = unmask_phrase(words[i], words[i + 1])
            if match:
                fixed[i], fixed[i + 1] = match
    return " ".join(head + fixed.get(i, unmask_word(core)) + tail
                    for i, (head, core, tail) in enumerate(tokens))


print(f"Vocabulary: {len(VOCAB)} words | bad words: {len(BAD_WORDS)} | bad phrases: {len(BAD_PHRASES)}\n")
for example in ["kutt*r bacc*", "কুত্*ার বাচ্চা", "কু*ার বা*া", "তুই একটা হা*ামি", "k-u-t-t-a"]:
    print(f"{example:20s} ->  {unmask_text(example)}")

In [ ]:
# ============================================================
# 2.3  Step 2 - Transliteration dictionary (learned from BanTH)
# ============================================================
# BanTH gives the same sentence in Romanized and in Bangla script. When both
# versions have the same number of words, word i of one matches word i of the
# other. Counting these matches gives a Roman -> Bangla word dictionary.

pair_counts = defaultdict(Counter)
for roman, bangla in zip(banth["roman"], banth["bangla"]):
    roman_words, bangla_words = split_words(clean_text(roman)), split_words(clean_text(bangla))
    if len(roman_words) == len(bangla_words):
        for r, b in zip(roman_words, bangla_words):
            if has_roman(r) and has_bangla(b):
                pair_counts[r][b] += 1
for r, b in SEED_ROMAN_TO_BANGLA.items():
    pair_counts[r][b] += 1000                      # the seed words always win

ROMAN_TO_BANGLA = {r: c.most_common(1)[0][0] for r, c in pair_counts.items()}

# Spelling key: people write the same Bangla word in many ways (bh/v, sh/s, double letters ...).
# Words with the same key are treated as the same word.
SPELLING_RULES = [("bh", "v"), ("sh", "s"), ("ph", "f"), ("kh", "k"), ("gh", "g"), ("th", "t"),
                  ("dh", "d"), ("ch", "c"), ("jh", "j"), ("z", "j"), ("w", "o"), ("y", "i"),
                  ("ee", "i"), ("oo", "u"), ("aa", "a")]

def spelling_key(word):
    """bhalobasi / valobashi / bhalobashi  ->  'valobasi'"""
    for old, new in SPELLING_RULES:
        word = word.replace(old, new)
    return re.sub(r"(.)\1+", r"\1", word)          # double letters -> single letter

key_counts = defaultdict(Counter)
for r, c in pair_counts.items():
    key_counts[spelling_key(r)].update(c)
KEY_TO_BANGLA = {k: c.most_common(1)[0][0] for k, c in key_counts.items()}

print("Dictionary:", len(ROMAN_TO_BANGLA), "Romanized words |", len(KEY_TO_BANGLA), "spelling keys")

# BanglaT5 is used only when the dictionary does not know a word (set in cell 2.4)
t5_transliterate = None        # set in 2.4 (FULL mode): one sentence at a time, used by the live demo
t5_transliterate_many = None   # set in 2.4 (FULL mode): many sentences in batches, used for the dataset


def transliterate_word(word):
    """One Romanized word -> Bangla, or None if unknown."""
    return ROMAN_TO_BANGLA.get(word) or KEY_TO_BANGLA.get(spelling_key(word))


def transliterate_text(text):
    """Romanized Bangla -> Bangla script. Bangla text is returned unchanged."""
    if not has_roman(text):
        return text
    output, unknown_words = [], 0
    for token in text.split():
        head, core, tail = split_punctuation(token)
        bangla = transliterate_word(core.lower()) if has_roman(core) else None
        if bangla:
            output.append(head + bangla + tail)
        else:
            unknown_words += has_roman(core)
            output.append(token)
    # Some words are still unknown -> BanglaT5 translates the whole (fully Romanized) sentence
    if unknown_words and t5_transliterate is not None and not has_bangla(text):
        return t5_transliterate(text)
    return " ".join(output)

for example in ["kuttar baccha", "Ami tomake bhalobasi", "tui ekta kharap manush"]:
    print(f"{example:25s} ->  {transliterate_text(example)}")

In [ ]:
# ============================================================
# 2.4  (FULL mode) Fine-tune BanglaT5 for transliteration
# ============================================================
# The dictionary cannot handle words it has never seen. BanglaT5 is a
# sequence-to-sequence model: it reads the Romanized sentence and writes the
# Bangla sentence. It is fine-tuned on BanTH sentence pairs.

T5_DIR        = os.path.join(OUT_DIR, "banglat5_transliterator_v2")   # v2: trained without the held-out pairs
T5_EPOCHS     = 2
T5_TRAIN_ROWS = 20000    # enough pairs for good results; keeps training time reasonable

def train_banglat5(pairs):
    t5_tok = AutoTokenizer.from_pretrained(T5_MODEL, use_fast=False)
    t5 = AutoModelForSeq2SeqLM.from_pretrained(T5_MODEL).to(DEVICE)
    optimizer = torch.optim.AdamW(t5.parameters(), lr=3e-4)
    t5.train()
    for epoch in range(T5_EPOCHS):
        pairs = pairs.sample(frac=1, random_state=epoch)       # shuffle every epoch
        total_loss, steps = 0.0, 0
        for i in range(0, len(pairs), 16):
            batch = pairs.iloc[i:i + 16]
            inputs = t5_tok(["transliterate: " + t for t in batch["roman"]], padding=True,
                            truncation=True, max_length=64, return_tensors="pt").to(DEVICE)
            labels = t5_tok(list(batch["bangla"]), padding=True, truncation=True,
                            max_length=64, return_tensors="pt").input_ids
            labels[labels == t5_tok.pad_token_id] = -100       # ignore padding in the loss
            loss = t5(**inputs, labels=labels.to(DEVICE)).loss
            loss.backward()
            optimizer.step(); optimizer.zero_grad()
            total_loss += loss.item(); steps += 1
        print(f"BanglaT5 epoch {epoch + 1}/{T5_EPOCHS}  loss = {total_loss / steps:.4f}")
    t5.save_pretrained(T5_DIR); t5_tok.save_pretrained(T5_DIR)
    return t5, t5_tok

def make_t5_transliterator(t5, t5_tok):
    t5.eval()
    def transliterate(text):
        inputs = t5_tok("transliterate: " + text, return_tensors="pt",
                        truncation=True, max_length=64).to(DEVICE)
        with torch.no_grad():
            output_ids = t5.generate(**inputs, max_new_tokens=64, num_beams=4)
        return t5_tok.decode(output_ids[0], skip_special_tokens=True)
    return transliterate

def make_t5_batch_transliterator(t5, t5_tok, batch_size=32):
    """The same model for many sentences at once (thousands of Romanized comments in the dataset):
    batches, a longer length limit and 2 beams keep this fast without cutting long comments short."""
    t5.eval()
    def transliterate_many(texts):
        outputs = []
        for i in range(0, len(texts), batch_size):
            inputs = t5_tok(["transliterate: " + t for t in texts[i:i + batch_size]], return_tensors="pt",
                            padding=True, truncation=True, max_length=128).to(DEVICE)
            with torch.no_grad():
                output_ids = t5.generate(**inputs, max_new_tokens=128, num_beams=2)
            outputs += t5_tok.batch_decode(output_ids, skip_special_tokens=True)
        return outputs
    return transliterate_many

if TRAIN_BANGLAT5:
    if os.path.isdir(T5_DIR):                                  # already trained in an earlier run
        t5_model = AutoModelForSeq2SeqLM.from_pretrained(T5_DIR).to(DEVICE)
        t5_tokenizer = AutoTokenizer.from_pretrained(T5_DIR, use_fast=False)
        print("BanglaT5 loaded from", T5_DIR)
    else:
        pairs = banth.assign(roman=banth["roman"].map(clean_text), bangla=banth["bangla"].map(clean_text))
        pairs = pairs[(pairs["roman"].str.len() > 0) & (pairs["roman"].str.len() < 200)]
        pairs = pairs.sample(min(T5_TRAIN_ROWS, len(pairs)), random_state=SEED)
        t5_model, t5_tokenizer = train_banglat5(pairs)
    t5_transliterate = make_t5_transliterator(t5_model, t5_tokenizer)
    t5_transliterate_many = make_t5_batch_transliterator(t5_model, t5_tokenizer)
    print("BanglaT5 test: 'Ami tomake bhalobasi' ->", t5_transliterate("Ami tomake bhalobasi"))
else:
    print("BanglaT5 skipped (QUICK mode). The dictionary + spelling keys are used for transliteration.")

In [ ]:
# ============================================================
# 2.5  The full normalisation pipeline   (examples from the problem statement)
# ============================================================
def normalize_text(text):
    """clean -> unmask -> transliterate. Returns the final text and every step."""
    steps = {"input": text}
    text = clean_text(text);          steps["cleaned"] = text
    text = unmask_text(text);         steps["unmasked"] = text
    text = transliterate_text(text);  steps["transliterated"] = text
    return text, steps

demo = pd.DataFrame([normalize_text(t)[1] for t in
                     ["kutt*r bacc*", "কুত্*ার বাচ্চা", "Ami tomake bhalobasi", "tui ekta h*rami"]])
display(demo)

# How often do hidden and Romanized comments appear in the real dataset?
n_masked = data["text"].str.contains(r"[^\s*]\*|\*[^\s*]").sum()     # a '*' inside a word
n_roman  = data["text"].map(lambda t: has_roman(t) and not has_bangla(t)).sum()
print(f"Comments with hidden letters (*): {n_masked} | fully Romanized comments: {n_roman}")

def normalize_dataset(texts):
    """normalize_text for the whole dataset. Fully Romanized comments that the dictionary cannot finish
    (e.g. the TB-OLID comments) go to BanglaT5 in batches - one by one would take very long."""
    global t5_transliterate
    saved, t5_transliterate = t5_transliterate, None          # first pass: dictionary + spelling keys only
    try:
        steps = [normalize_text(t)[1] for t in texts]
    finally:
        t5_transliterate = saved
    results = [s["transliterated"] for s in steps]
    todo = [i for i, s in enumerate(steps) if has_roman(s["transliterated"]) and not has_bangla(s["unmasked"])]
    if todo and t5_transliterate_many is not None:
        print(f"BanglaT5: transliterating {len(todo)} Romanized comments in batches ...")
        for i, bangla in zip(todo, t5_transliterate_many([steps[i]["unmasked"] for i in todo])):
            results[i] = bangla
    return results

# Normalise the whole forensic dataset the same way (so training and later use match)
data["clean_text"] = normalize_dataset(list(data["text"]))
data = data[data["clean_text"].str.len() > 0].reset_index(drop=True)
print("Rows after normalisation:", len(data))

### 2.6 Quantitative evaluation of the normalisation

The examples above show single successes. To measure the pipeline, **300 evaluation items** are generated from the
**held-out BanTH pairs** (1.3), which the dictionary, the vocabulary and BanglaT5 have never seen. The correct
output of every item is known exactly (it is the Bangla side of the pair), so the evaluation does not depend on
manual labels:

| Category | Input | Reference |
|---|---|---|
| `masked` | one word per sentence hidden with `*` (half in Bangla script, half Romanized); a known bad word is hidden when the sentence contains one | Bangla sentence |
| `banglish` | the Romanized sentence | Bangla sentence |
| `mixed` | first half in Bangla script, second half Romanized; one Romanized word spelled with a separator (`-`, `/` or `.`) | Bangla sentence |

**Metrics.** *Exact match*: the output has exactly the words of the reference. *Word-level accuracy*: matched words
(longest common subsequence) divided by the length of the longer word sequence, so missing and extra words both
count as errors. For masked items, the unmasking step alone is also scored in the item's own script.

**Systems.** *Dictionary only* (dictionary + spelling keys) versus the *hybrid* pipeline (dictionary + BanglaT5 for
fully Romanized sentences with unknown words). Failures are listed with their likely cause.

In [ ]:
# ============================================================
# 2.6  Quantitative evaluation of the normalisation (held-out items)
# ============================================================
from difflib import SequenceMatcher

SEPARATORS = ["-", "/", "."]     # '-' and '/' are joined by the unmasker; '.' is not (tests a known limitation)

def mask_word(word, rng):
    """Hide 1-2 inner characters with '*' (the first and the last character stay visible)."""
    n = 1 if len(word) < 6 else rng.choice([1, 2])
    start = rng.randint(1, len(word) - 1 - n)
    return word[:start] + "*" * n + word[start + n:]

WORD_PATTERN = {"bangla": r"[ঀ-৿]{4,}", "roman": r"[A-Za-z]{4,}"}   # letters only (no emoji, digits)

def hide_one_word(sentence, rng, script):
    """Hide one word of the sentence: a known bad word if there is one, else a random word of 4+ letters."""
    tokens = sentence.split()
    parts = [split_punctuation(t) for t in tokens]
    usable = [i for i, (_, core, _) in enumerate(parts) if re.fullmatch(WORD_PATTERN[script], core)]
    if not usable:
        return None, None
    bad = [i for i in usable if parts[i][1].lower() in BAD_WORDS]
    i = rng.choice(bad or usable)
    head, core, tail = parts[i]
    tokens[i] = head + mask_word(core, rng) + tail
    return " ".join(tokens), core

def build_norm_eval(pairs, seed=7):
    rng = random.Random(seed)
    pairs = pairs.assign(roman=pairs["roman"].map(clean_text), bangla=pairs["bangla"].map(clean_text))   # no emoji / links
    pairs = pairs.sample(frac=1, random_state=seed).reset_index(drop=True)
    same_length = pairs["roman"].str.split().str.len() == pairs["bangla"].str.split().str.len()
    mixed_pool = pairs[same_length].head(len(pairs) // 3)
    rest = pairs.drop(mixed_pool.index)
    masked_pool, banglish_pool = rest.iloc[: len(rest) // 2], rest.iloc[len(rest) // 2:]
    items = []
    for k, r in enumerate(masked_pool.itertuples()):
        script, sentence = ("bangla", r.bangla) if k % 2 == 0 else ("roman", r.roman)
        text, hidden = hide_one_word(sentence, rng, script)
        if text:
            items.append({"category": "masked", "variant": f"{script} script", "input": text, "gold": r.bangla,
                          "unmask_gold": sentence, "hidden_word": hidden})
    for r in banglish_pool.itertuples():
        items.append({"category": "banglish", "variant": "roman script", "input": r.roman, "gold": r.bangla})
    for k, r in enumerate(mixed_pool.itertuples()):
        bangla_words, roman_words = r.bangla.split(), r.roman.split()
        half = len(bangla_words) // 2
        tokens = bangla_words[:half] + roman_words[half:]
        sep = SEPARATORS[k % len(SEPARATORS)]
        usable = [i for i in range(half, len(tokens)) if re.fullmatch(r"[A-Za-z]{3,8}", tokens[i])]
        if usable:
            i = rng.choice(usable)
            tokens[i] = sep.join(tokens[i])
        items.append({"category": "mixed", "variant": f"separator '{sep}'" if usable else "mixed script only",
                      "input": " ".join(tokens), "gold": r.bangla})
    return pd.DataFrame(items)

def words_of(text):
    return split_words(clean_text(text))

def word_accuracy(pred, gold):
    p, g = words_of(pred), words_of(gold)
    if not p and not g:
        return 1.0
    matched = sum(m.size for m in SequenceMatcher(None, p, g, autojunk=False).get_matching_blocks())
    return matched / max(len(p), len(g))

def run_normaliser(texts, use_t5):
    """normalize_text for a list of texts, with or without the BanglaT5 fallback."""
    global t5_transliterate
    saved = t5_transliterate
    if not use_t5:
        t5_transliterate = None
    try:
        return [normalize_text(t)[0] for t in texts]
    finally:
        t5_transliterate = saved

norm_eval = build_norm_eval(banth_heldout) if len(banth_heldout) else pd.DataFrame()
NORM_SYSTEMS = {"dictionary only": False}
if t5_transliterate is not None:
    NORM_SYSTEMS["dictionary + BanglaT5"] = True

if len(norm_eval):
    for name, use_t5 in NORM_SYSTEMS.items():
        outputs = run_normaliser(list(norm_eval["input"]), use_t5)
        norm_eval[f"output: {name}"] = outputs
        norm_eval[f"exact: {name}"] = [words_of(o) == words_of(g) for o, g in zip(outputs, norm_eval["gold"])]
        norm_eval[f"word acc: {name}"] = [word_accuracy(o, g) for o, g in zip(outputs, norm_eval["gold"])]
    masked = norm_eval["category"] == "masked"
    norm_eval["unmask exact"] = np.nan
    norm_eval.loc[masked, "unmask exact"] = [
        float(words_of(unmask_text(clean_text(t))) == words_of(g))
        for t, g in zip(norm_eval.loc[masked, "input"], norm_eval.loc[masked, "unmask_gold"])]

    def summarise(groups):
        rows = []
        for key, g in groups:
            row = {"items": len(g)}
            for name in NORM_SYSTEMS:
                row[f"exact match ({name})"] = g[f"exact: {name}"].mean()
                row[f"word accuracy ({name})"] = g[f"word acc: {name}"].mean()
            row["unmasking alone, exact"] = g["unmask exact"].mean()
            rows.append(pd.Series(row, name=key))
        return pd.DataFrame(rows).round(3)

    norm_summary = pd.concat([summarise(norm_eval.groupby("category")),
                              summarise([("ALL", norm_eval)])])
    norm_by_variant = summarise(norm_eval.groupby(["category", "variant"]))
    print("Normalisation on held-out items (higher is better):")
    display(norm_summary)
    display(norm_by_variant)

    # Failure analysis of the final (hybrid) pipeline
    final_system = list(NORM_SYSTEMS)[-1]
    def failure_cause(row):
        output = row[f"output: {final_system}"]
        if row["category"] == "masked" and row["unmask exact"] == 0:
            return "unmasking: hidden word not in the vocabulary, or another word fits the pattern"
        if row["category"] == "mixed" and re.search(r"[A-Za-z][.\-/][A-Za-z]", output):
            return "separator not recognised"
        if has_roman(output):
            return "unseen Romanized word (left in Latin script)"
        return "other spelling or word choice than the reference (ambiguous spelling / context)"
    failures = norm_eval[~norm_eval[f"exact: {final_system}"]].copy()
    failures["cause"] = failures.apply(failure_cause, axis=1)
    print(f"\nFailures of '{final_system}': {len(failures)} of {len(norm_eval)} items")
    print(failures["cause"].value_counts().to_string())
    with pd.option_context("display.max_colwidth", 70):
        display(failures.groupby("cause").head(3)[["category", "input", f"output: {final_system}", "gold", "cause"]])

    norm_eval.to_csv(os.path.join(RUN_DIR, "normalisation_eval_items.csv"), index=False)
    norm_summary.to_csv(os.path.join(RUN_DIR, "normalisation_eval_summary.csv"))
    norm_by_variant.to_csv(os.path.join(RUN_DIR, "normalisation_eval_by_variant.csv"))
    failures["cause"].value_counts().to_csv(os.path.join(RUN_DIR, "normalisation_failure_causes.csv"))
else:
    norm_summary = pd.DataFrame()
    print("No held-out BanTH pairs (BanTH download failed) - normalisation evaluation skipped.")

---
# Section 3 · Cryptographic hash-linked evidence ledger (audit trail)

The evidence store is a **prototype cryptographic hash-linked ledger** kept as one local JSON file. It is **not** a
distributed blockchain: there is one writer, no network, no consensus protocol and no independent copy of the data.

**Structure.** Every block holds `index`, a UTC `timestamp`, a `type`, its `data`, `previous_hash` (the hash of the
previous block), a proof-of-work `nonce` and its own `hash`.

**Hashing.** `hash = SHA-256(canonical JSON of index, timestamp, type, data, previous_hash, nonce)` with sorted keys,
so the same content always gives the same hash. Model weights and evidence texts are stored as SHA-256 fingerprints.
A block is accepted only when its hash starts with `00` (proof-of-work, difficulty 2).

**Verification.** `verify()` recomputes every hash and checks the proof-of-work prefix, the link to the previous block
and the index sequence. `checkpoint()` writes the current length and head hash to a **separate file**; in practice
this value is handed to an independent party (supervisor, court registry) or published. `verify_checkpoints()`
checks that every recorded head is still in the chain, unchanged.

**Security assumptions and limits.** SHA-256 is assumed to be collision-resistant and the checkpoints are assumed to be
stored outside the control of the ledger holder. The ledger detects a change to any stored block, a changed timestamp
and a removed block. It does **not** stop the holder of the file from rewriting the chain: with difficulty 2 a block is
re-mined in milliseconds, so a coordinated rewrite of all later blocks, or cutting off the newest blocks, passes
`verify()`. Only an independent checkpoint detects this (tamper experiments, 9.1). The ledger therefore provides
**tamper-evident records and chain-of-custody documentation**; it does not by itself establish legal admissibility.

Four kinds of content blocks follow the `GENESIS` block:

| Block type | Stored when |
|---|---|
| `CLIENT_UPDATE` | a federated client finishes local training (fingerprint of its model weights) |
| `GLOBAL_MODEL` | the server finishes FedAvg (fingerprint + validation scores) |
| `EVIDENCE` | a comment is classified as harmful (with enough confidence) |
| `USER_PROFILE` | a user account profile is created (requirement, Section 8) |

In [ ]:
# ============================================================
# 3.1  The hash-linked evidence ledger
# ============================================================
def sha256(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def model_fingerprint(state_dict):
    """SHA-256 of all model weights. Any change to any weight gives a different value."""
    h = hashlib.sha256()
    for name in sorted(state_dict):
        h.update(name.encode())
        h.update(state_dict[name].detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()


class HashLinkedLedger:
    """Prototype cryptographic hash-linked evidence ledger: one local JSON file, one writer, no consensus.
    Every block stores the SHA-256 hash of the previous block, so a change to a stored block is detected.
    The small proof-of-work only slows rewriting down; an external checkpoint detects a complete rewrite."""
    DIFFICULTY = 2      # a valid block hash must start with this many zeros

    def __init__(self, file_path):
        self.file_path = file_path
        self.checkpoint_path = file_path.replace(".json", "_checkpoints.json")
        self.chain, self.checkpoints = [], []
        self.add_block("GENESIS", {"message": "Bangla social-media forensic ledger"})

    @staticmethod
    def compute_hash(block):
        content = {k: block[k] for k in ["index", "timestamp", "type", "data", "previous_hash", "nonce"]}
        return sha256(json.dumps(content, sort_keys=True, ensure_ascii=False, default=str))

    def mine(self, block):
        """Proof-of-work: change the nonce until the hash starts with "00"."""
        block["nonce"] = 0
        block["hash"] = self.compute_hash(block)
        while not block["hash"].startswith("0" * self.DIFFICULTY):
            block["nonce"] += 1
            block["hash"] = self.compute_hash(block)

    def add_block(self, block_type, data):
        block = {
            "index": len(self.chain),
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "type": block_type,
            "data": copy.deepcopy(data),   # a private copy: later changes to `data` cannot alter the block
            "previous_hash": self.chain[-1]["hash"] if self.chain else "0" * 64,
        }
        self.mine(block)
        self.chain.append(block)
        self.save()
        return block

    def verify(self):
        """Re-check every hash, proof-of-work, link and index. Returns (is_valid, message)."""
        for i, block in enumerate(self.chain):
            if self.compute_hash(block) != block["hash"]:
                return False, f"Block {i} was changed (hash does not match its data)"
            if not block["hash"].startswith("0" * self.DIFFICULTY):
                return False, f"Block {i} has no valid proof-of-work"
            if i > 0 and block["previous_hash"] != self.chain[i - 1]["hash"]:
                return False, f"Block {i} is not linked to block {i - 1}"
            if block["index"] != i:
                return False, f"Block at position {i} carries index {block['index']} (a block was removed)"
        return True, f"All {len(self.chain)} blocks are valid"

    def checkpoint(self, label):
        """Record the current head (length + hash of the last block) OUTSIDE the ledger file.
        In practice this value is published or handed to an independent party."""
        cp = {"label": label, "length": len(self.chain), "head_hash": self.chain[-1]["hash"],
              "time": datetime.now(timezone.utc).isoformat()}
        self.checkpoints.append(cp)
        if self.file_path:
            with open(self.checkpoint_path, "w", encoding="utf-8") as f:
                json.dump(self.checkpoints, f, indent=2)
        return cp

    def verify_checkpoints(self, checkpoints=None):
        """Every recorded head must still be in the chain, at the same position, with the same hash."""
        checkpoints = self.checkpoints if checkpoints is None else checkpoints
        for cp in checkpoints:
            n = cp["length"]
            if len(self.chain) < n:
                return False, f"Checkpoint '{cp['label']}': the chain has {len(self.chain)} blocks, {n} were recorded"
            if self.chain[n - 1]["hash"] != cp["head_hash"]:
                return False, f"Checkpoint '{cp['label']}': block {n - 1} differs from the recorded head hash"
        return True, f"All {len(checkpoints)} checkpoints match"

    def remine_from(self, start):
        """ATTACK SIMULATION ONLY (9.1): what the holder of the file could do to hide a change -
        renumber, relink and re-mine every block from `start` to the end."""
        for i in range(start, len(self.chain)):
            block = self.chain[i]
            block["index"] = i
            block["previous_hash"] = self.chain[i - 1]["hash"] if i else "0" * 64
            self.mine(block)

    def save(self):
        if self.file_path:
            with open(self.file_path, "w", encoding="utf-8") as f:
                json.dump(self.chain, f, ensure_ascii=False, indent=2, default=str)

    def count_by_type(self):
        return Counter(b["type"] for b in self.chain)


ledger = HashLinkedLedger(os.path.join(RUN_DIR, "forensic_ledger.json"))
print(json.dumps(ledger.chain[0], indent=2))

---
# Section 4 · Novel part – fixing the class imbalance (Problem 1)

1. **Fixed split (made once in `build_dataset.py`):** 4,000 test and 2,000 validation comments, stratified by source
   and label. They keep the **natural** class mix, are **never** filtered or balanced, and are identical for dataset
   versions A, B and C, so the versions can be compared fairly. The validation set selects the best federated round
   and tunes the thresholds; the test set is used once at the end.
2. **Over-sampling (training set only, inside every client):** every comment is kept, and the comments of the rare
   harmful classes are repeated until each is as large as the largest harmful class (`OVERSAMPLE_TO = 1.0`), but no
   class is repeated more than 3 times over (`MAX_OVERSAMPLE`), so a tiny class is not simply memorised.
   Each client does this with **its own data**, so no comment moves between clients.
   Over-sampling is preferred to under-sampling `normal`: in preliminary runs, under-sampling discarded about
   11,000 real `normal` training comments without improving macro F1.
3. **No class-weighted loss** (`USE_CLASS_WEIGHTS = False`). In preliminary runs, class weights on top of
   re-sampling corrected the imbalance twice: the model labelled too many comments as harmful (recall far above
   precision), `normal` lost F1 and macro F1 did not improve. When enabled, the weights are
   `sqrt(total / (number_of_classes × class_count))`, computed by each federated client from **its own data only**.

In [ ]:
# ============================================================
# 4.1  Train / validation / test split (fixed in the dataset file)
# ============================================================
train_df, val_df, test_df = (data[data["split"] == s].reset_index(drop=True) for s in ("train", "val", "test"))
print(f"Train: {len(train_df)} | Validation: {len(val_df)} | Test: {len(test_df)}")


In [ ]:
# ============================================================
# 4.2  Client data (IID / Non-IID) + over-sampling inside every client (+ optional class weights)
# ============================================================
OVERSAMPLE_TO     = 1.0     # repeat rare harmful classes until they are as big as the biggest harmful class
MAX_OVERSAMPLE    = 3       # ... but never more than 3 x the class's own size
USE_CLASS_WEIGHTS = False   # re-sampling + class weights together corrected the imbalance twice
                            # (too many false 'harmful' predictions), so only over-sampling is used

# 'source' partition: one client per platform group, i.e. every organisation holds the data of its own platform
SOURCE_GROUPS = {
    "YouTube":            ["banhate", "vitd", "vulgar"],
    "Facebook":           ["boc", "tbolid"],
    "other social media": ["bdshs", "belal"],
}

def split_into_clients(df, n_clients, partition="iid", seed=SEED, alpha=DIRICHLET_ALPHA):
    """Give each client its own part of the data.
    iid       : sort by label and deal the rows out like cards -> every client has a similar class mix
    dirichlet : for every class, the shares of the clients are drawn from Dirichlet(alpha) -> different class mixes
    source    : one client per platform group (SOURCE_GROUPS) -> different sources and class mixes"""
    df = df.sample(frac=1, random_state=seed)
    if partition == "iid":
        df = df.sort_values("label", kind="stable")
        return [df.iloc[i::n_clients].reset_index(drop=True) for i in range(n_clients)]
    if partition == "source":
        groups = list(SOURCE_GROUPS.values())
        assert len(groups) == n_clients, "SOURCE_GROUPS needs one group per client"
        clients = [df[df["source"].isin(g)].reset_index(drop=True) for g in groups]
        assert all(len(c) for c in clients), "a platform group has no training data in this dataset version"
        return clients
    if partition == "dirichlet":
        rng = np.random.default_rng(seed)
        for _ in range(100):                     # redraw until every client holds at least 5% of the data
            parts = [[] for _ in range(n_clients)]
            for _, rows in df.groupby("label"):
                bounds = [0, *(np.cumsum(rng.dirichlet([alpha] * n_clients))[:-1] * len(rows)).astype(int), len(rows)]
                for k in range(n_clients):
                    parts[k].append(rows.iloc[bounds[k]:bounds[k + 1]])
            sizes = [sum(len(p) for p in client) for client in parts]
            if min(sizes) >= 0.05 * len(df):
                break
        return [pd.concat(p).sample(frac=1, random_state=seed).reset_index(drop=True) for p in parts]
    raise ValueError(f"unknown partition '{partition}'")

def balance_training_data(df, seed=SEED):
    """Keep EVERY comment and repeat the comments of each rare harmful class until it has
    OVERSAMPLE_TO x (biggest harmful class) comments (at most MAX_OVERSAMPLE x its own size).
    No comment is thrown away."""
    counts = df.loc[df["label"] != "normal", "label"].value_counts()
    target = int(OVERSAMPLE_TO * counts.max())
    parts = [df]
    for label, n in counts.items():
        goal = min(target, MAX_OVERSAMPLE * n)
        if n < goal:
            rows = df[df["label"] == label]
            copies, rest = divmod(goal - n, n)            # whole copies + a random part
            parts += [rows] * copies + [rows.sample(rest, random_state=seed)]
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

def class_weights(labels):
    """Rare classes get a bigger weight (square root of the inverse frequency)."""
    counts = np.bincount([LABEL2ID[l] for l in labels], minlength=len(LABELS))
    weights = np.sqrt(len(labels) / (len(LABELS) * np.maximum(counts, 1)))
    return torch.tensor(weights, dtype=torch.float)

# The main runs use the IID split. The same clients are used by both runs.
# Each client balances ITS OWN data (the data never leaves the client).
baseline_clients  = split_into_clients(train_df, NUM_CLIENTS, "iid")
balanced_clients  = [balance_training_data(c) for c in baseline_clients]
balanced_train_df = pd.concat(balanced_clients, ignore_index=True)   # all clients together, for the table

# Compare before / after
before = train_df["label"].value_counts().reindex(LABELS, fill_value=0)
after  = balanced_train_df["label"].value_counts().reindex(LABELS, fill_value=0)
balance_table = pd.DataFrame({"before": before, "after": after}, index=LABELS)
if USE_CLASS_WEIGHTS:
    balance_table["class_weight"] = class_weights(balanced_train_df["label"]).numpy().round(2)
print(balance_table)
def imbalance(counts):
    """How many times more 'normal' comments than the smallest harmful class."""
    return counts["normal"] / max(counts.drop("normal").min(), 1)

print(f"\nImbalance (normal / smallest class): {imbalance(before):.1f}x -> {imbalance(after):.1f}x")

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(LABELS))
ax.bar(x - 0.2, before.values, width=0.4, label="Before (original)", color="#95A5A6")
ax.bar(x + 0.2, after.values,  width=0.4, label="After over-sampling", color="#2980B9")
ax.set_xticks(x); ax.set_xticklabels(LABELS)
ax.set_ylabel("Training comments"); ax.set_title("Training data before and after balancing")
ax.legend()
save_fig(fig, "2_balancing")
plt.show()

---
# Section 5 · Simulated federated learning classifier

**Idea:** several organisations (clients) train one model together **without sharing their raw comments**.
In this notebook the clients are **simulated** on one GPU.

In every **round**:
1. The server sends the current **global model** to every client.
2. Each client trains it on **its own data** for a few epochs.
3. Each client sends back only its **model weights** (never the data). A fingerprint is stored in the ledger.
4. The server averages the weights (**FedAvg**, bigger clients count more) → new global model, also stored in the ledger.

After the last round, the global model of the round with the **best validation macro F1** is kept.
All clients use the same model (**BanglaBERT** + a classification layer), so the averaging is meaningful.

**Client data (`split_into_clients`, 4.2):**

| Partition | How the training set is divided | Class mix per client |
|---|---|---|
| `iid` (main runs) | sorted by label and dealt out like cards | about the same for all clients |
| `dirichlet` (Non-IID, Section 10) | for every class, the client shares are drawn from Dirichlet(α = 0.5) | very different |
| `source` (Non-IID, Section 10) | one client per platform group: YouTube (BanHate, VITD, vulgar corpus), Facebook (BOC, TB-OLID), other social media (BD-SHS, Belal et al.) | different sources and classes |

**Centralized reference (5.3):** the same BanglaBERT trained on all training comments in one place with the same number
of passes over the data, so the cost of federated training can be measured (10.3).

**Privacy note.** Only model weights leave a client, but shared weights and updates can still leak information about
the training data (gradient inversion, membership inference). Differential privacy and secure aggregation are **not**
implemented here; the notebook demonstrates data locality, not a formal privacy guarantee.

In [ ]:
# ============================================================
# 5.1  Helpers: data loaders, local training, FedAvg, prediction
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(CLASSIFIER_MODEL)

def make_loader(df, shuffle):
    """Convert comments + labels into batches of token ids."""
    enc = tokenizer(list(df["clean_text"]), truncation=True, padding="max_length",
                    max_length=MAX_LEN, return_tensors="pt")
    labels = torch.tensor([LABEL2ID[l] for l in df["label"]])
    return DataLoader(TensorDataset(enc["input_ids"], enc["attention_mask"], labels),
                      batch_size=BATCH_SIZE, shuffle=shuffle)

def new_model():
    return AutoModelForSequenceClassification.from_pretrained(
        CLASSIFIER_MODEL, num_labels=len(LABELS),
        id2label=dict(enumerate(LABELS)), label2id=LABEL2ID).to(DEVICE)

def train_local(model, loader, weights=None):
    """One client trains the model on its own data. Returns the average loss."""
    loss_fn = torch.nn.CrossEntropyLoss(weight=None if weights is None else weights.to(DEVICE))
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)      # needed for mixed precision
    model.train()
    total_loss, steps = 0.0, 0
    for epoch in range(LOCAL_EPOCHS):
        for input_ids, mask, labels in loader:
            input_ids, mask, labels = input_ids.to(DEVICE), mask.to(DEVICE), labels.to(DEVICE)
            with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=USE_AMP):
                logits = model(input_ids=input_ids, attention_mask=mask).logits
            loss = loss_fn(logits.float(), labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # avoid too-large updates
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad()
            total_loss += loss.item(); steps += 1
    return total_loss / max(steps, 1)

def fed_avg(client_states, client_sizes):
    """FedAvg: new weight = sum(client weight x client_size / total_size)."""
    total = sum(client_sizes)
    averaged = {}
    for name, first in client_states[0].items():
        if first.is_floating_point():
            averaged[name] = sum(state[name] * (size / total)
                                 for state, size in zip(client_states, client_sizes))
        else:
            averaged[name] = first      # integer buffers (e.g. position ids) are not averaged
    return averaged

@torch.no_grad()
def predict_proba(model, texts, batch_size=64):
    """Return class probabilities for a list of (already normalised) texts."""
    model.eval()
    all_probs = []
    for i in range(0, len(texts), batch_size):
        enc = tokenizer(list(texts[i:i + batch_size]), truncation=True, padding=True,
                        max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=USE_AMP):
            logits = model(**enc).logits
        all_probs.append(torch.softmax(logits.float(), dim=-1).cpu())
    return torch.cat(all_probs).numpy()

def adjust_probs(probs, bias=None):
    """Move the decision threshold of every class: add a per-class bias to the log-probabilities
    and renormalise. bias = None leaves the probabilities unchanged (see 6.2)."""
    if bias is None:
        return probs
    scores = np.log(np.clip(probs, 1e-9, 1.0)) + bias
    scores = np.exp(scores - scores.max(axis=1, keepdims=True))
    return scores / scores.sum(axis=1, keepdims=True)

def evaluate(model, df, bias=None):
    """Accuracy, macro F1 / precision / recall and F1 of every class on a labelled DataFrame."""
    y_true = [LABEL2ID[l] for l in df["label"]]
    y_pred = adjust_probs(predict_proba(model, list(df["clean_text"])), bias).argmax(axis=1)
    per_class = f1_score(y_true, y_pred, labels=range(len(LABELS)), average=None, zero_division=0)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, labels=range(len(LABELS)), average="macro", zero_division=0),
        "macro_precision": precision_score(y_true, y_pred, labels=range(len(LABELS)), average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, labels=range(len(LABELS)), average="macro", zero_division=0),
        "per_class_f1": dict(zip(LABELS, per_class.round(4))),
        "y_true": y_true, "y_pred": y_pred,
    }

print("Helpers ready.")

In [ ]:
# ============================================================
# 5.2  The federated training loop
# ============================================================
def run_federated(clients, use_class_weights, run_name, ledger_=None):
    """Train one global model with FedAvg on the clients' own datasets (see 4.2).
    Every step is recorded in the ledger (the main ledger unless ledger_ is given).
    Returns the global model of the round with the best validation macro F1."""
    ledger_ = ledger if ledger_ is None else ledger_
    print(f"\n========== Federated training: {run_name} ==========")
    loaders = [make_loader(c, shuffle=True) for c in clients]
    for i, c in enumerate(clients, 1):
        print(f"Client {i}: {len(c)} comments  {c['label'].value_counts().to_dict()}")

    global_model = new_model()
    history, best = [], {"macro_f1": -1.0, "round": 0, "state": None}
    for rnd in range(1, FL_ROUNDS + 1):
        states, sizes = [], []
        for cid, (client_df, loader) in enumerate(zip(clients, loaders), 1):
            local_model = copy.deepcopy(global_model)          # 1. client receives the global model
            weights = class_weights(client_df["label"]) if use_class_weights else None
            loss = train_local(local_model, loader, weights)   # 2. trains on its own data
            state = {k: v.detach().cpu().clone() for k, v in local_model.state_dict().items()}
            states.append(state); sizes.append(len(client_df)) # 3. sends back only the weights
            ledger_.add_block("CLIENT_UPDATE", {
                "run": run_name, "round": rnd, "client": cid, "samples": len(client_df),
                "train_loss": round(loss, 4), "weights_hash": model_fingerprint(state)})
            print(f"  Round {rnd} | client {cid} | loss = {loss:.4f}")
            del local_model
            if DEVICE == "cuda":
                torch.cuda.empty_cache()

        global_model.load_state_dict(fed_avg(states, sizes))  # 4. server averages (FedAvg)
        del states

        result = evaluate(global_model, val_df)                # validation set picks the best round
        ledger_.add_block("GLOBAL_MODEL", {
            "run": run_name, "round": rnd, "val_accuracy": round(float(result["accuracy"]), 4),
            "val_macro_f1": round(float(result["macro_f1"]), 4),
            "weights_hash": model_fingerprint(global_model.state_dict())})
        history.append({"round": rnd, "accuracy": result["accuracy"],
                        "macro_f1": result["macro_f1"], **result["per_class_f1"]})
        print(f"Round {rnd} done | validation accuracy = {result['accuracy']:.4f} "
              f"| validation macro F1 = {result['macro_f1']:.4f}")
        if result["macro_f1"] > best["macro_f1"]:
            best = {"macro_f1": result["macro_f1"], "round": rnd,
                    "state": {k: v.detach().cpu().clone() for k, v in global_model.state_dict().items()}}

    global_model.load_state_dict(best["state"])
    print(f"-> Using the model of round {best['round']} (best validation macro F1 = {best['macro_f1']:.4f})")
    return global_model, pd.DataFrame(history)

In [ ]:
# ============================================================
# 5.3  Centralized reference (no federated learning)
# ============================================================
def run_centralized(train_data, run_name, ledger_=None, use_class_weights=False):
    """ONE model trained on all training comments in one place. The number of passes over the data equals the
    federated setup (FL_ROUNDS x LOCAL_EPOCHS) and the epoch with the best validation macro F1 is kept, exactly as
    the best federated round is kept. The optimiser is restarted every epoch, like a federated client every round."""
    ledger_ = ledger if ledger_ is None else ledger_
    print(f"\n========== Centralized training: {run_name} ==========")
    loader = make_loader(train_data, shuffle=True)
    central_model = new_model()
    weights = class_weights(train_data["label"]) if use_class_weights else None
    history, best = [], {"macro_f1": -1.0, "round": 0, "state": None}
    for epoch in range(1, FL_ROUNDS + 1):
        loss = train_local(central_model, loader, weights)
        result = evaluate(central_model, val_df)
        ledger_.add_block("GLOBAL_MODEL", {
            "run": run_name, "round": epoch, "training": "centralized", "train_loss": round(loss, 4),
            "val_accuracy": round(float(result["accuracy"]), 4), "val_macro_f1": round(float(result["macro_f1"]), 4),
            "weights_hash": model_fingerprint(central_model.state_dict())})
        history.append({"round": epoch, "accuracy": result["accuracy"],
                        "macro_f1": result["macro_f1"], **result["per_class_f1"]})
        print(f"Epoch {epoch} | loss = {loss:.4f} | validation macro F1 = {result['macro_f1']:.4f}")
        if result["macro_f1"] > best["macro_f1"]:
            best = {"macro_f1": result["macro_f1"], "round": epoch,
                    "state": {k: v.detach().cpu().clone() for k, v in central_model.state_dict().items()}}
    central_model.load_state_dict(best["state"])
    print(f"-> Using the model of epoch {best['round']} (best validation macro F1 = {best['macro_f1']:.4f})")
    return central_model, pd.DataFrame(history)

---
# Section 6 · Novel part – results: per-class F1 (Problem 2)

Two models are trained with exactly the same federated setup:

| Run | Training data | Loss | Decision |
|---|---|---|---|
| **Baseline** | original (imbalanced) | standard cross-entropy | class with the highest probability |
| **Improved** | over-sampled rare harmful classes | standard cross-entropy | per-class thresholds tuned on the validation set |

**Threshold tuning (6.2):** a model trained on imbalanced data assigns too much probability to `normal`. Every class
receives a small bias that shifts its decision threshold. The biases are chosen on the **validation set** to maximise
macro F1; the test set is never used for this, so the test scores remain unbiased.

Both models are evaluated on the **same untouched test set**. Table 6.3 also reports the improved model *before*
threshold tuning (`oversampled_F1`), so the contribution of each step is visible.

**Seeds and statistics (Section 10):** every run stores its scores and test predictions in `runs_v2/` in the
output folder. Section 10 repeats baseline and improved with seeds 43 and 44, reports the mean ± standard deviation
over seeds 42, 43 and 44 and a paired bootstrap confidence interval for every seed. A difference is treated as
meaningful only when it clearly exceeds the spread between seeds.
Cell 6.5 also reports the scores per source and on the **checked test labels**: all 4,000 test comments re-labelled
blind against the label guideline (single LLM annotator; see `dataset/DATASHEET.md`).

**Metric:** macro F1 (the unweighted mean F1 over the five classes) is the main metric, because accuracy is dominated
by the `normal` class. Identical runs differ by about 0.002–0.005 macro F1 (GPU non-determinism), so smaller
differences are not interpreted as improvements. The confusion matrices in 6.4 show which classes are confused.

In [ ]:
# ============================================================
# 6.1  Train the baseline and the improved model
# ============================================================
RUN_BASELINE = True     # False = train only the improved model (Section 10 always trains both)

if RUN_BASELINE:
    baseline_model, baseline_history = run_federated(baseline_clients, use_class_weights=False, run_name="baseline")
    baseline_result = evaluate(baseline_model, test_df)
    del baseline_model                         # free GPU memory before the next run
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

model, improved_history = run_federated(balanced_clients, use_class_weights=USE_CLASS_WEIGHTS, run_name="improved")
oversampled_result = evaluate(model, test_df)     # before threshold tuning (6.2)

In [ ]:
# ============================================================
# 6.2  Tune the decision thresholds on the VALIDATION set
# ============================================================
def tune_class_bias(probs, y_true, grid=np.linspace(-2, 2, 41), passes=3):
    """Find one bias per class that gives the highest macro F1.
    Coordinate search: try every grid value for one class at a time, keep the best, repeat."""
    def macro_f1(bias):
        y_pred = adjust_probs(probs, bias).argmax(axis=1)
        return f1_score(y_true, y_pred, labels=range(len(LABELS)), average="macro", zero_division=0)
    bias = np.zeros(len(LABELS))
    best = macro_f1(bias)
    for _ in range(passes):
        for k in range(1, len(LABELS)):          # 'normal' stays at 0: only the differences matter
            for value in grid:
                trial = bias.copy(); trial[k] = value
                score = macro_f1(trial)
                if score > best:
                    best, bias = score, trial
    return bias, best

val_probs = predict_proba(model, list(val_df["clean_text"]))
val_true  = [LABEL2ID[l] for l in val_df["label"]]
untuned_val_f1 = f1_score(val_true, val_probs.argmax(axis=1), labels=range(len(LABELS)),
                          average="macro", zero_division=0)
CLASS_BIAS, tuned_val_f1 = tune_class_bias(val_probs, val_true)

print("Bias per class (log scale, > 0 = predicted more often):")
print(pd.Series(CLASS_BIAS.round(2), index=LABELS).to_string())
print(f"\nValidation macro F1: {untuned_val_f1:.4f} -> {tuned_val_f1:.4f}")

improved_result = evaluate(model, test_df, bias=CLASS_BIAS)   # the final model, used from here on

# Save the final classifier together with its class biases
final_dir = os.path.join(RUN_DIR, "final_classifier")
model.save_pretrained(final_dir)
tokenizer.save_pretrained(final_dir)
with open(os.path.join(final_dir, "class_bias.json"), "w") as f:
    json.dump(dict(zip(LABELS, CLASS_BIAS.round(4).tolist())), f, indent=2)
print("\nFinal classifier + class biases saved to", final_dir)
ledger.checkpoint("after federated training")    # head hash kept OUTSIDE the ledger file (3.1)

In [ ]:
# ============================================================
# 6.3  Per-class F1 table and chart (baseline vs improved, TEST set)
# ============================================================
columns = {"oversampled_F1": oversampled_result, "improved_F1": improved_result}
if RUN_BASELINE:
    columns = {"baseline_F1": baseline_result, **columns}
table = pd.DataFrame({name: r["per_class_f1"] for name, r in columns.items()}).reindex(LABELS)
table.loc["MACRO F1"] = [round(r["macro_f1"], 4) for r in columns.values()]
table.loc["ACCURACY"] = [round(r["accuracy"], 4) for r in columns.values()]
table.loc["MACRO PRECISION"] = [round(r["macro_precision"], 4) for r in columns.values()]
table.loc["MACRO RECALL"] = [round(r["macro_recall"], 4) for r in columns.values()]
if RUN_BASELINE:
    table["change"] = (table["improved_F1"] - table["baseline_F1"]).round(4)
table.to_csv(os.path.join(RUN_DIR, "per_class_f1.csv"))
display(table)
print("oversampled = over-sampling only | improved = over-sampling + tuned thresholds (final model)")

if RUN_BASELINE:
    better = [l for l in LABELS if table.loc[l, "change"] > 0]
    print(f"F1 improved for {len(better)} of {len(LABELS)} classes: {better}")
    print(f"Macro F1: baseline {baseline_result['macro_f1']:.4f} -> over-sampling {oversampled_result['macro_f1']:.4f} "
          f"-> + thresholds {improved_result['macro_f1']:.4f}")
    print("(Accuracy can go down a little when the rare harmful classes are predicted more often.)")

print("\nDetailed report (improved model, test set):")
print(classification_report(improved_result["y_true"], improved_result["y_pred"],
                            labels=range(len(LABELS)), target_names=LABELS, zero_division=0))

names  = {"baseline_F1": "Baseline", "oversampled_F1": "Over-sampling",
          "improved_F1": "Over-sampling + thresholds"}
colors = {"baseline_F1": "#95A5A6", "oversampled_F1": "#85C1E9", "improved_F1": "#27AE60"}
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(LABELS))
width = 0.8 / len(columns)
for j, (name, r) in enumerate(columns.items()):
    bars = ax.bar(x + (j - (len(columns) - 1) / 2) * width, [r["per_class_f1"][l] for l in LABELS], width,
                  label=names[name], color=colors[name])
    ax.bar_label(bars, fmt="%.2f", fontsize=7)
ax.set_xticks(x); ax.set_xticklabels(LABELS); ax.set_ylim(0, 1)
ax.set_ylabel("F1 score (test set)"); ax.set_title("Problem 2: F1 score of every class")
ax.legend()
save_fig(fig, "3_per_class_f1")
plt.show()

In [ ]:
# ============================================================
# 6.4  Macro F1 per federated round + confusion matrices
# ============================================================
n_plots = 3 if RUN_BASELINE else 2
fig, axes = plt.subplots(1, n_plots, figsize=(6 * n_plots, 4.5))

if RUN_BASELINE:
    axes[0].plot(baseline_history["round"], baseline_history["macro_f1"], "o-", label="Baseline", color="#95A5A6")
axes[0].plot(improved_history["round"], improved_history["macro_f1"], "o-", label="Improved", color="#27AE60")
axes[0].set_xlabel("Federated round"); axes[0].set_ylabel("Macro F1 (validation)")
axes[0].set_title("Macro F1 after each round"); axes[0].legend()
axes[0].set_xticks(improved_history["round"])

results_to_plot = ([("Baseline", baseline_result)] if RUN_BASELINE else []) + [("Improved", improved_result)]
for ax, (name, result) in zip(axes[1:], results_to_plot):
    cm = confusion_matrix(result["y_true"], result["y_pred"], labels=range(len(LABELS)))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=LABELS, yticklabels=LABELS, ax=ax, cbar=False)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"Confusion matrix ({name}, test set)")
fig.tight_layout()
save_fig(fig, "4_rounds_and_confusion_matrix")
plt.show()

In [ ]:
# ============================================================
# 6.5  Scores per source + on the checked test labels, and save this run
# ============================================================
RUNS_DIR = os.path.join(OUT_DIR, "runs_v2")    # v2 run files (with test predictions); older files in runs/ are not mixed in
os.makedirs(RUNS_DIR, exist_ok=True)

def scores(y_true, y_pred, labels=range(len(LABELS))):
    return {"rows": len(y_true), "accuracy": round(float(accuracy_score(y_true, y_pred)), 4),
            "macro_f1": round(float(f1_score(y_true, y_pred, labels=list(labels), average="macro", zero_division=0)), 4)}

def per_source_scores(y_true, y_pred):
    """Macro F1 per source, over the classes that source contains (e.g. VITD has only normal / violence)."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    result = {}
    for source in sorted(test_df["source"].unique()):
        rows = (test_df["source"] == source).to_numpy()
        result[source] = scores(y_true[rows], y_pred[rows], labels=sorted(set(y_true[rows])))
    return result

# Checked labels: the test labels checked against the guideline (dataset/test_label_check.csv)
has_check = test_df["checked_label"].isin(LABELS).to_numpy() if "checked_label" in test_df else np.zeros(len(test_df), bool)

def checked_scores(y_true, y_pred):
    if not has_check.any():
        return {}
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    y_checked = test_df.loc[has_check, "checked_label"].map(LABEL2ID).to_numpy()
    result = scores(y_checked, y_pred[has_check])
    result["per_class_f1"] = dict(zip(LABELS, f1_score(y_checked, y_pred[has_check], labels=range(len(LABELS)),
                                                       average=None, zero_division=0).round(4).tolist()))
    result["source_label_agreement"] = round(float((y_checked == y_true[has_check]).mean()), 4)
    result["macro_f1_source_labels_same_rows"] = scores(y_true[has_check], y_pred[has_check])["macro_f1"]
    return result

def to_plain(r):
    return {**{k: round(float(r[k]), 4) for k in ("macro_f1", "accuracy", "macro_precision", "macro_recall")},
            "per_class_f1": {k: float(v) for k, v in r["per_class_f1"].items()}}

def save_run(seed, training, partition, baseline_res, oversampled_res, improved_res, bias, histories, clients):
    """One JSON file per configuration: scores, test predictions, training curves and client class mixes."""
    run = {"data_version": DATA_VERSION, "seed": seed, "run_mode": RUN_MODE, "training": training,
           "partition": partition, "train_rows": len(train_df), "fl_rounds": FL_ROUNDS,
           "client_sizes": [len(c) for c in clients] if clients else None,
           "client_label_counts": ([c["label"].value_counts().reindex(LABELS, fill_value=0).astype(int).tolist()
                                    for c in clients] if clients else None),
           "baseline": to_plain(baseline_res) if baseline_res else None,
           "oversampled_only": to_plain(oversampled_res), "improved": to_plain(improved_res),
           "class_bias": dict(zip(LABELS, np.round(bias, 3).tolist())),
           "history": {k: {"round": h["round"].astype(int).tolist(), "macro_f1": h["macro_f1"].round(4).tolist()}
                       for k, h in histories.items()},
           "test_ids": test_df["id"].astype(str).tolist(),
           "test_true": [int(v) for v in improved_res["y_true"]],
           "test_pred": {"baseline": [int(v) for v in baseline_res["y_pred"]] if baseline_res else None,
                         "improved": [int(v) for v in improved_res["y_pred"]]},
           "per_source": per_source_scores(improved_res["y_true"], improved_res["y_pred"]),
           "checked": checked_scores(improved_res["y_true"], improved_res["y_pred"])}
    path = os.path.join(RUNS_DIR, f"{DATA_VERSION}_{training}_{partition}_seed{seed}_{RUN_MODE}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(run, f)
    return path

# (a) Per source
per_source = per_source_scores(improved_result["y_true"], improved_result["y_pred"])
print("Improved model, test set, per source:")
display(pd.DataFrame(per_source).T)

# (b) Checked labels
checked_result = checked_scores(improved_result["y_true"], improved_result["y_pred"])
if checked_result:
    print(f"\nChecked test labels ({has_check.sum()} rows): macro F1 = {checked_result['macro_f1']:.4f} "
          f"(source labels: {checked_result['macro_f1_source_labels_same_rows']:.4f} on the same rows)")
    print(f"The source label agrees with the checked label for {checked_result['source_label_agreement']:.1%} of these rows.")
    print(pd.Series(checked_result["per_class_f1"]).to_string())
else:
    print("\nNo checked test labels in the dataset file yet.")

# (c) Save this run (seed SEED, federated, IID), so Section 10 can compare seeds and settings
histories = {"improved": improved_history, **({"baseline": baseline_history} if RUN_BASELINE else {})}
run_path = save_run(SEED, "federated", "iid", baseline_result if RUN_BASELINE else None,
                    oversampled_result, improved_result, CLASS_BIAS, histories, baseline_clients)
print("\nRun saved:", run_path)

---
# Section 7 · Novel part – end-to-end analysis of a comment

`analyze_texts()` runs the full chain: **clean → unmask → transliterate → classify → (if harmful) record in the ledger**.
The classifier uses the class thresholds tuned in 6.2, so the confidence below is the probability after that adjustment.

* A harmful prediction with confidence ≥ `EVIDENCE_MIN_CONFIDENCE` becomes an **EVIDENCE** block. The block stores the
  original text, its SHA-256 fingerprint, the normalised text, the prediction and the confidence.
* A harmful prediction with lower confidence is marked **uncertain** (for a human to review) and is not saved.

In [ ]:
# ============================================================
# 7.1  Analyse texts and record evidence   (examples from the problem statement)
# ============================================================
def analyze_texts(texts, save_to_ledger=True, context=None):
    """Run the whole pipeline on a list of texts.
    context: optional list of dicts (one per text) with extra info, e.g. user id and time."""
    texts = [str(t) for t in texts]
    normalized, steps = [], []
    for t in texts:
        final_text, s = normalize_text(t)
        normalized.append(final_text); steps.append(s)
    probs = adjust_probs(predict_proba(model, normalized), CLASS_BIAS)   # thresholds tuned in 6.2

    rows = []
    for i, text in enumerate(texts):
        label = LABELS[int(probs[i].argmax())]
        confidence = round(float(probs[i].max()), 4)
        is_harmful = label != "normal" and confidence >= EVIDENCE_MIN_CONFIDENCE
        decision = ("normal" if label == "normal" else
                    "EVIDENCE" if is_harmful else "uncertain (review)")
        row = {**steps[i], "prediction": label, "confidence": confidence,
               "decision": decision, "is_harmful": is_harmful, "block_index": None}
        if save_to_ledger and is_harmful:
            block = ledger.add_block("EVIDENCE", {
                "original_text": text,
                "text_sha256": sha256(text),          # proves the text was not changed later
                "normalized_text": normalized[i],
                "prediction": label,
                "confidence": confidence,
                **(context[i] if context else {"source": "manual_input"}),
            })
            row["block_index"] = block["index"]
        rows.append(row)
    return pd.DataFrame(rows).astype({"block_index": "Int64"})   # Int64 = whole numbers, empty allowed

examples = ["kutt*r bacc*", "কুত্*ার বাচ্চা", "Ami tomake bhalobasi", "আজ আবহাওয়া খুব সুন্দর"]
demo_results = analyze_texts(examples)
display(demo_results.drop(columns=["cleaned", "is_harmful"]))
print("Blocks in the ledger:", dict(ledger.count_by_type()))

# Show one evidence block exactly as it is stored in the ledger
evidence_index = demo_results["block_index"].dropna()
if len(evidence_index):
    print("\nEvidence block as stored in the ledger:")
    print(json.dumps(ledger.chain[int(evidence_index.iloc[0])], ensure_ascii=False, indent=2))

In [ ]:
# ============================================================
# 7.2  Interactive analysis of a single comment
# ============================================================
# The input may be Bangla or Romanized Bangla and may contain words hidden with '*', e.g.:
#   "tui ekta sh*la"   |   "এই শু*রের বাচ্চা"   |   "kemon acho bhai"
MY_COMMENT = "tui ekta sh*la"          # @param {type:"string"}
SAVE_AS_EVIDENCE = False               # @param {type:"boolean"}

live = analyze_texts([MY_COMMENT], save_to_ledger=SAVE_AS_EVIDENCE).iloc[0]
print("Input          :", live["input"])
print("Unmasked       :", live["unmasked"])
print("Transliterated :", live["transliterated"])
print("Prediction     :", live["prediction"], f"(confidence {live['confidence']:.2f})")
print("Decision       :", live["decision"],
      f"-> saved as block #{live['block_index']}" if pd.notna(live["block_index"]) else "")

---
# Section 8 · Requirement: DF analysis of a Facebook user account

This section implements the requirement: it monitors the activities of a **public** account (status, share, comment) as **time-series data**, keeps its public **About / personal info**, and builds a **profile with 5 aspects**. Every activity is first processed by the novel pipeline (Sections 2 and 7), so hidden and Romanized harmful comments are also detected.

```
Public Facebook account
 ├── About: public profile information (visibility, joined, followers ...)
 └── Activities with date/time  ──►  TIME-SERIES DATA
       • status  (the user writes a post)
       • share   (the user shares someone else's post)
       • comment (the user comments on other people's posts)
              │  every activity: clean → unmask → transliterate → classify
              ▼
 PROFILING with 5 aspects  ──►  profile vector  ──►  risk level (Low / Medium / High)
              │
              ▼
 LEDGER: one EVIDENCE block per harmful activity + one USER_PROFILE block
```

**Input format.** A CSV file with the columns `user_id, timestamp, activity_type, text`
(`activity_type` is `status`, `share` or `comment`). The file is selected with `USER_ACTIVITY_CSV` in cell 8.1.

**Ethics.** Only **public** data, collected lawfully (e.g. by an investigator with legal authority). This notebook does
**not** scrape Facebook. Without a CSV it creates **simulated** accounts from real test-set comments, clearly marked as simulated.

**Simulated accounts (12 behaviour patterns, 12 weeks each):**

| User | Kind | Behaviour built into the simulation | Expected result |
|---|---|---|---|
| `user_A` | benign | ordinary user, about 5% offensive posts | **Low** risk |
| `user_B` | risky | harmful posts grow from 5% to 70% | **escalating** trend |
| `user_C` | risky | half of the posts harmful, mostly comments on others, uses hidden words | **harasser** role |
| `user_D` | benign | no harmful posts | **Low** risk |
| `user_E` | benign | no harmful posts, very active (15 per week) | **Low** risk |
| `user_F` | benign | mostly comments, about 5% offensive | **Low** risk |
| `user_G` | benign | borderline, about 10% offensive / cyberbully | **Low** risk |
| `user_H` | risky | harmful posts fall from 60% to 5% | **decreasing** trend |
| `user_I` | risky | 40% harmful, mostly shared hate speech | **spreader** role |
| `user_J` | risky | 70% violence / hate speech throughout | **High** risk |
| `user_K` | risky | low activity (1 per week), 30% cyberbully | **Medium** risk |
| `user_L` | rare-severe | about 8% violence | **Low** risk by design (see 8.6) |

In [ ]:
# ============================================================
# 8.1  Load (or simulate) the public activity of user accounts
# ============================================================
USER_ACTIVITY_CSV = None     # e.g. "user_activity.csv" ; None = use simulated accounts

# Public "About" information of each account (context in the report, not used in the score)
USER_ABOUT = {}

def simulate_user(user_id, weeks, posts_per_week, harm_start, harm_end, harm_types, type_probs, extra=()):
    """Create a SIMULATED public timeline from real test-set comments.
    harm_start / harm_end = chance that a post is harmful in the first / last week
    (for example 0.05 -> 0.70 gives an 'escalating' user)."""
    user_seed = SEED + sum(map(ord, user_id))              # same result every run
    rng, np_rng = random.Random(user_seed), np.random.default_rng(user_seed)
    texts_by_label = {l: test_df.loc[test_df["label"] == l, "text"].tolist() for l in LABELS}
    start = datetime(2026, 1, 5)
    rows = []
    for week in range(weeks):
        p_harm = harm_start + (harm_end - harm_start) * week / max(weeks - 1, 1)
        for _ in range(np_rng.poisson(posts_per_week)):     # number of posts this week
            label = rng.choice(harm_types) if rng.random() < p_harm else "normal"
            pool = texts_by_label[label] or texts_by_label["normal"]
            rows.append({
                "user_id": user_id,
                "timestamp": start + timedelta(weeks=week, hours=rng.randint(0, 7 * 24 - 1)),
                "activity_type": rng.choices(["status", "share", "comment"], weights=type_probs)[0],
                "text": rng.choice(pool),
                "true_label": label,          # known only because the data is simulated
            })
    # Extra (e.g. obfuscated) comments placed in the last week
    for text in extra:
        rows.append({"user_id": user_id, "timestamp": start + timedelta(weeks=weeks - 1, hours=rng.randint(0, 150)),
                     "activity_type": "comment", "text": text, "true_label": "offensive"})
    return pd.DataFrame(rows)

# user, kind, description, posts/week, harm start, harm end, harm types, status/share/comment, extra, expected result
ACCOUNTS = [
    ("user_A", "benign", "ordinary, ~5% offensive", 5, 0.05, 0.05, ["offensive"], [0.5, 0.3, 0.2], (),
     ("risk level", "Low")),
    ("user_B", "risky", "escalating 5% -> 70%", 5, 0.05, 0.70, ["offensive", "hate_speech", "violence"],
     [0.4, 0.4, 0.2], (), ("trend", "escalating")),
    ("user_C", "risky", "harasser, 50% harmful comments", 5, 0.50, 0.50, ["cyberbully", "offensive"],
     [0.1, 0.1, 0.8], ("kutt*r bacc*", "কুত্*ার বাচ্চা"), ("role", "harasser (harmful comments on others)")),
    ("user_D", "benign", "no harmful posts", 5, 0.0, 0.0, ["offensive"], [0.6, 0.3, 0.1], (),
     ("risk level", "Low")),
    ("user_E", "benign", "no harmful posts, very active", 15, 0.0, 0.0, ["offensive"], [0.3, 0.3, 0.4], (),
     ("risk level", "Low")),
    ("user_F", "benign", "commenter, ~5% offensive", 5, 0.05, 0.05, ["offensive"], [0.1, 0.1, 0.8], (),
     ("risk level", "Low")),
    ("user_G", "benign", "borderline, ~10% offensive/cyberbully", 5, 0.10, 0.10, ["offensive", "cyberbully"],
     [0.4, 0.3, 0.3], (), ("risk level", "Low")),
    ("user_H", "risky", "de-escalating 60% -> 5%", 5, 0.60, 0.05, ["cyberbully", "hate_speech"],
     [0.4, 0.3, 0.3], (), ("trend", "decreasing")),
    ("user_I", "risky", "spreader, 40% shared hate speech", 5, 0.40, 0.40, ["hate_speech"], [0.1, 0.8, 0.1], (),
     ("role", "spreader (shares harmful content)")),
    ("user_J", "risky", "persistent severe, 70% violence/hate", 5, 0.70, 0.70, ["violence", "hate_speech"],
     [0.5, 0.3, 0.2], (), ("risk level", "High")),
    ("user_K", "risky", "low activity (1/week), 30% cyberbully", 1, 0.30, 0.30, ["cyberbully"], [0.4, 0.3, 0.3], (),
     ("risk level", "Medium")),
    ("user_L", "rare-severe", "rare but severe, ~8% violence", 5, 0.08, 0.08, ["violence"], [0.5, 0.3, 0.2], (),
     ("risk level", "Low")),
]

if USER_ACTIVITY_CSV:
    activity = pd.read_csv(USER_ACTIVITY_CSV, parse_dates=["timestamp"])
    missing = {"user_id", "timestamp", "activity_type", "text"} - set(activity.columns)
    assert not missing, f"The CSV is missing these columns: {missing}"
    activity["activity_type"] = activity["activity_type"].str.strip().str.lower()
    SIMULATED_EXPECTATION, ACCOUNT_KIND = {}, {}
else:
    print(f"Using {len(ACCOUNTS)} SIMULATED accounts (no real user data).")
    activity = pd.concat([simulate_user(uid, 12, ppw, h0, h1, types, probs, extra)
                          for uid, _, _, ppw, h0, h1, types, probs, extra, _ in ACCOUNTS], ignore_index=True)
    about_rng = random.Random(SEED)
    USER_ABOUT = {uid: {"profile": "public", "joined": f"{about_rng.randint(2012, 2025)}-{about_rng.randint(1, 12):02d}",
                        "followers": about_rng.randint(20, 3000), "simulated": True, "behaviour": desc}
                  for uid, _, desc, *_ in ACCOUNTS}
    # What the simulation was built to show (checked automatically in 8.3)
    SIMULATED_EXPECTATION = {a[0]: a[-1] for a in ACCOUNTS}
    ACCOUNT_KIND = {a[0]: a[1] for a in ACCOUNTS}

activity = activity.sort_values(["user_id", "timestamp"]).reset_index(drop=True)
print("Activities per user and type:")
print(activity.groupby(["user_id", "activity_type"]).size().unstack(fill_value=0))
activity.head()

In [ ]:
# ============================================================
# 8.2  Analyse every activity (harmful ones become EVIDENCE blocks)
# ============================================================
context = [{"user_id": r.user_id, "activity_type": r.activity_type, "timestamp": str(r.timestamp)}
           for r in activity.itertuples()]
results = analyze_texts(activity["text"].tolist(), save_to_ledger=True, context=context)

activity["normalized_text"] = results["transliterated"].values
activity["prediction"]      = results["prediction"].values
activity["confidence"]      = results["confidence"].values
activity["decision"]        = results["decision"].values
activity["is_harmful"]      = results["is_harmful"].values
activity["block_index"]     = results["block_index"].values
activity["week"]            = activity["timestamp"].dt.to_period("W").dt.start_time   # time-series step

print(f"Activities analysed: {len(activity)} | harmful (evidence): {int(activity['is_harmful'].sum())} "
      f"| uncertain: {int((activity['decision'] == 'uncertain (review)').sum())}")

# With simulated data the true label is known, so we can check the detection
if "true_label" in activity:
    truly_harmful = activity["true_label"] != "normal"
    print(f"Check on simulated data: harmful / normal decided correctly for "
          f"{(truly_harmful == activity['is_harmful']).mean():.1%} of the activities")
    print(pd.crosstab(truly_harmful.map({True: "truly harmful", False: "truly normal"}),
                      activity["is_harmful"].map({True: "detected harmful", False: "detected normal"})))

activity[["user_id", "timestamp", "activity_type", "normalized_text", "prediction", "decision", "block_index"]].tail(8)

In [ ]:
# ============================================================
# 8.3  Build the forensic profile (5 aspects)
# ============================================================
# Severity of each class (0 = harmless ... 4 = most serious). An analyst can change these.
SEVERITY = {"normal": 0, "offensive": 1, "cyberbully": 2, "hate_speech": 3, "violence": 4}

RISK_WEIGHTS = (0.5, 0.3, 0.2)   # harmful ratio, harm load (ratio x severity), rising trend - justified and tested in 8.6

def risk_score(harm_ratio, severity, rise, weights=RISK_WEIGHTS):
    """Risk score (0-100) = 100 x (w1 x ratio + w2 x ratio x severity + w3 x positive rise).
    Harm load = harmful ratio x severity, so a user with only a few harmful posts gets only
    a few points for severity, even when those posts are serious. All three terms lie in [0, 1] and the
    weights sum to 1, so the score lies in [0, 100] and never decreases when a term grows."""
    w_ratio, w_load, w_trend = weights
    return 100 * (w_ratio * harm_ratio + w_load * harm_ratio * severity + w_trend * min(max(rise, 0.0), 1.0))

# Cut-offs = the score of a stable reference user (no rise) whose harmful posts have medium severity (0.5):
#   Medium from 20% harmful activity - a lower ratio is close to the share of normal comments that the
#          classifier itself flags as harmful (measured in 8.6), so it cannot be told apart from its mistakes;
#   High   from 50% harmful activity - half of everything the user posts.
REFERENCE_SEVERITY = 0.5
MEDIUM_RATIO, HIGH_RATIO = 0.20, 0.50

def risk_cutoffs(weights=RISK_WEIGHTS, medium_ratio=MEDIUM_RATIO, high_ratio=HIGH_RATIO):
    return (risk_score(medium_ratio, REFERENCE_SEVERITY, 0.0, weights),
            risk_score(high_ratio, REFERENCE_SEVERITY, 0.0, weights))

RISK_MEDIUM, RISK_HIGH = risk_cutoffs()    # 13.0, 32.5

ROLE = {"status": "author (writes harmful posts)",
        "share": "spreader (shares harmful content)",
        "comment": "harasser (harmful comments on others)"}

def build_user_profile(user_df, weights=RISK_WEIGHTS, cutoffs=None):
    harmful = user_df[user_df["is_harmful"]]
    weekly = user_df.groupby("week").agg(posts=("is_harmful", "size"), harmful=("is_harmful", "sum"))
    weekly["harm_ratio"] = weekly["harmful"] / weekly["posts"]

    # Aspect 1: activity level - how many posts per week, and at which hour most often
    posts_per_week = weekly["posts"].mean()
    peak_hour = int(user_df["timestamp"].dt.hour.mode()[0])

    # Aspect 2: harmful ratio - share of all activities that are harmful
    harm_ratio = user_df["is_harmful"].mean()

    # Aspect 3: harm type and severity (0..1)
    harm_types = {k: int(v) for k, v in harmful["prediction"].value_counts().items()}
    main_type = harmful["prediction"].mode()[0] if len(harmful) else "none"
    severity = harmful["prediction"].map(SEVERITY).mean() / max(SEVERITY.values()) if len(harmful) else 0.0

    # Aspect 4: time trend = straight line through the weekly harmful ratio.
    # "rise" = how much the ratio grew from the first to the last week.
    slope = np.polyfit(np.arange(len(weekly)), weekly["harm_ratio"], 1)[0] if len(weekly) > 1 else 0.0
    rise = slope * (len(weekly) - 1)
    trend = "escalating" if rise > 0.15 else ("decreasing" if rise < -0.15 else "stable")

    # Aspect 5: behaviour channel - where does the harmful content appear?
    channel = harmful["activity_type"].value_counts(normalize=True).round(2).to_dict()
    role = ROLE.get(max(channel, key=channel.get), "none") if channel else "none"

    risk = risk_score(harm_ratio, severity, rise, weights)
    medium, high = cutoffs or (RISK_MEDIUM, RISK_HIGH)
    level = "High" if risk >= high else ("Medium" if risk >= medium else "Low")

    return {
        "1_activity":  {"posts_per_week": round(float(posts_per_week), 2), "peak_hour": peak_hour,
                        "total_activities": int(len(user_df))},
        "2_harmful_ratio": round(float(harm_ratio), 3),
        "3_harm_type": {"main_type": main_type, "counts": harm_types, "severity": round(float(severity), 3)},
        "4_trend":     {"trend": trend, "rise": round(float(rise), 3)},
        "5_channel":   {"share_by_type": channel, "role": role},
        "profile_vector": [round(float(v), 3) for v in
                           [posts_per_week, harm_ratio, severity, rise,
                            channel.get("status", 0), channel.get("share", 0), channel.get("comment", 0)]],
        "risk_score": round(float(risk), 1),
        "risk_level": level,
        "weekly": weekly,
    }

profiles = {uid: build_user_profile(udf) for uid, udf in activity.groupby("user_id")}

# Summary table: one row per user
summary = pd.DataFrame([{
    "user": uid,
    "joined": USER_ABOUT.get(uid, {}).get("joined", "?"),
    "followers": USER_ABOUT.get(uid, {}).get("followers", "?"),
    "posts/week": p["1_activity"]["posts_per_week"],
    "harmful ratio": p["2_harmful_ratio"],
    "main harm type": p["3_harm_type"]["main_type"],
    "severity": p["3_harm_type"]["severity"],
    "trend": p["4_trend"]["trend"],
    "role": p["5_channel"]["role"],
    "risk score": p["risk_score"],
    "risk level": p["risk_level"],
} for uid, p in profiles.items()]).set_index("user")

# Simulated data: does the profile show what the simulation was built to show?
if SIMULATED_EXPECTATION:
    summary.insert(0, "kind", [ACCOUNT_KIND.get(u, "") for u in summary.index])
    summary["true harmful ratio"] = (activity.groupby("user_id")["true_label"]
                                     .apply(lambda s: (s != "normal").mean()).round(3))
    summary["expected (simulation)"] = [" = ".join(SIMULATED_EXPECTATION[u]) for u in summary.index]
    summary["found?"] = ["yes" if summary.loc[u, SIMULATED_EXPECTATION[u][0]] == SIMULATED_EXPECTATION[u][1]
                         else "no" for u in summary.index]

summary.to_csv(os.path.join(RUN_DIR, "user_profiles_summary.csv"))
display(summary)

In [ ]:
# ============================================================
# 8.4  Time-series charts for every user
# ============================================================
for uid, user_df in activity.groupby("user_id"):
    weekly_counts = (user_df.assign(shown=np.where(user_df["is_harmful"], user_df["prediction"], "normal"))
                     .groupby(["week", "shown"]).size().unstack(fill_value=0)
                     .reindex(columns=LABELS, fill_value=0))
    weekly = profiles[uid]["weekly"]

    fig, axes = plt.subplots(1, 2, figsize=(14, 3.8))
    weekly_counts.index = weekly_counts.index.strftime("%d %b")
    weekly_counts.plot(kind="bar", stacked=True, ax=axes[0], width=0.8,
                       color=[CLASS_COLORS[c] for c in LABELS])
    axes[0].set_title(f"{uid}: weekly activities by detected class")
    axes[0].set_xlabel("Week"); axes[0].set_ylabel("Activities")
    axes[0].set_ylim(0, weekly_counts.sum(axis=1).max() * 1.3)    # space for the legend
    axes[0].legend(fontsize=8, ncol=5, loc="upper left")

    x = np.arange(len(weekly))
    axes[1].plot(x, weekly["harm_ratio"], "o-", color="#C0392B", label="Harmful ratio")
    if len(weekly) > 1:
        a, b = np.polyfit(x, weekly["harm_ratio"], 1)
        axes[1].plot(x, a * x + b, "--", color="black", label="Trend")
    axes[1].set_ylim(0, 1); axes[1].set_xlabel("Week number"); axes[1].set_ylabel("Harmful ratio")
    axes[1].set_title(f"{uid}: trend = {profiles[uid]['4_trend']['trend']}, "
                      f"risk = {profiles[uid]['risk_level']} ({profiles[uid]['risk_score']})")
    axes[1].legend()
    save_fig(fig, f"5_timeline_{uid}")
    plt.show()

# Risk score of all users
fig, ax = plt.subplots(figsize=(max(6, 0.9 * len(summary)), 3.8))
colors = {"Low": "#27AE60", "Medium": "#E1B12C", "High": "#C0392B"}
bars = ax.bar(summary.index, summary["risk score"], color=[colors[l] for l in summary["risk level"]])
ax.bar_label(bars, labels=[f"{s} ({l})" for s, l in zip(summary["risk score"], summary["risk level"])])
ax.axhline(RISK_MEDIUM, ls=":", color="gray"); ax.axhline(RISK_HIGH, ls=":", color="gray")
ax.set_ylim(0, max(50, summary["risk score"].max() + 10)); ax.set_ylabel("Risk score")
ax.tick_params(axis="x", rotation=45)
ax.set_title(f"User risk scores (Low < {RISK_MEDIUM:g} ≤ Medium < {RISK_HIGH:g} ≤ High)")
save_fig(fig, "6_user_risk_scores")
plt.show()

In [ ]:
# ============================================================
# 8.5  Store every profile in the ledger + print the forensic report
# ============================================================
profile_records = {}
for uid, p in profiles.items():
    user_df = activity[activity["user_id"] == uid]
    record = {
        "user_id": uid,
        "about": USER_ABOUT.get(uid, {}),
        "period": f"{user_df['timestamp'].min():%Y-%m-%d} to {user_df['timestamp'].max():%Y-%m-%d}",
        **{k: v for k, v in p.items() if k != "weekly"},
        "evidence_blocks": user_df["block_index"].dropna().astype(int).tolist(),  # links to EVIDENCE blocks
    }
    block = ledger.add_block("USER_PROFILE", record)
    record["profile_block"] = block["index"]
    profile_records[uid] = record

    a, h, t, c = p["1_activity"], p["3_harm_type"], p["4_trend"], p["5_channel"]
    print("=" * 72)
    print(f"FORENSIC PROFILE: {uid}      (ledger block #{block['index']}, hash {block['hash'][:16]}...)")
    print("=" * 72)
    print("About            :", ", ".join(f"{k}: {v}" for k, v in record["about"].items()) or "-")
    print("Period           :", record["period"])
    print(f"1. Activity      : {a['total_activities']} activities, {a['posts_per_week']} per week, "
          f"most active at {a['peak_hour']:02d}:00")
    print(f"2. Harmful ratio : {p['2_harmful_ratio']:.1%} of the activities are harmful")
    print(f"3. Harm type     : mostly {h['main_type']}  {h['counts']}  (severity {h['severity']})")
    print(f"4. Trend         : {t['trend']}  (harmful ratio changed by {t['rise']:+.2f} over the period)")
    print(f"5. Channel       : {c['role']}  {c['share_by_type']}")
    print("Profile vector   :", p["profile_vector"], " [posts/week, harm ratio, severity, rise, status, share, comment]")
    print(f"RISK             : {p['risk_score']} -> {p['risk_level']}")
    print(f"Evidence blocks  : {len(record['evidence_blocks'])} blocks, e.g. {record['evidence_blocks'][:10]}")
    print()

ledger.checkpoint("after user profiles")        # head hash kept OUTSIDE the ledger file (3.1)

with open(os.path.join(RUN_DIR, "user_profiles.json"), "w", encoding="utf-8") as f:
    json.dump(profile_records, f, ensure_ascii=False, indent=2, default=str)

### 8.6 Justification of the risk score and sensitivity analysis

**Mathematical basis.** Risk = 100 · ( w₁·r + w₂·r·s + w₃·max(0, min(rise, 1)) ), with *r* = harmful ratio,
*s* = mean severity of the harmful activities (0–1) and *rise* = change of the weekly harmful ratio over the period
(straight-line fit). Every term lies in [0, 1] and w₁ + w₂ + w₃ = 1, so the score is a weighted average of three bounded
indicators: it lies between 0 and 100 and never decreases when *r*, *s* or *rise* increases.

* **r, w₁ = 0.5** – how much of the account is harmful: the most direct and most stable signal, so it gets half the weight.
* **r·s, w₂ = 0.3** – harm load: severity counts in proportion to how often harm occurs, so a few severe posts do not
  dominate the score.
* **rise, w₃ = 0.2** – escalation: the smallest weight, because a slope over twelve weekly ratios is the noisiest signal;
  only increases add risk.

**Cut-offs.** Both are the score of a stable reference user with medium severity (*s* = 0.5): Medium from *r* = 0.20,
High from *r* = 0.50. The Medium cut-off must lie above the share of normal comments that the classifier flags as
harmful; otherwise ordinary accounts reach Medium through classifier errors alone.

The cell below (i) measures that false-positive share, (ii) computes the profiles from the **true** labels of the
simulated activities (formula only, no classifier errors), (iii) varies the weights and (iv) varies the cut-offs. For
every setting it counts the **benign accounts rated Medium or High** (false-positive risk assignments) and the
**risky accounts rated Low** (missed). All results are simulation-based and need validation on real accounts.

In [ ]:
# ============================================================
# 8.6  Sensitivity analysis of the risk score (weights, cut-offs, classifier errors)
# ============================================================
normal_rows = np.array(improved_result["y_true"]) == LABEL2ID["normal"]
fpr_normal = float((np.asarray(improved_result["y_pred"])[normal_rows] != LABEL2ID["normal"]).mean())
print(f"(i) Normal TEST comments predicted as harmful: {fpr_normal:.1%} "
      f"(the Medium cut-off assumes a harmful ratio of {MEDIUM_RATIO:.0%})")

if "true_label" in activity and ACCOUNT_KIND:
    truly_normal = activity["true_label"] == "normal"
    fpr_activity = float(activity.loc[truly_normal, "is_harmful"].mean())
    print(f"    Truly normal simulated activities recorded as harmful (confidence >= {EVIDENCE_MIN_CONFIDENCE}): "
          f"{fpr_activity:.1%}")

    BENIGN = [u for u, k in ACCOUNT_KIND.items() if k == "benign"]
    RISKY  = [u for u, k in ACCOUNT_KIND.items() if k == "risky"]
    RARE   = [u for u, k in ACCOUNT_KIND.items() if k == "rare-severe"]
    oracle = activity.assign(is_harmful=activity["true_label"] != "normal", prediction=activity["true_label"])

    def levels_for(frame, weights=RISK_WEIGHTS, medium_ratio=MEDIUM_RATIO, high_ratio=HIGH_RATIO):
        cut = risk_cutoffs(weights, medium_ratio, high_ratio)
        return pd.Series({u: build_user_profile(g, weights, cut)["risk_level"] for u, g in frame.groupby("user_id")})

    def assess(levels):
        return {"benign rated Medium/High": int(levels[BENIGN].isin(["Medium", "High"]).sum()),
                "risky rated Low": int((levels[RISKY] == "Low").sum()),
                "rare-severe rated Low": int((levels[RARE] == "Low").sum())}

    # (ii) the formula alone (true labels) versus the full pipeline (classifier)
    base_true, base_pred = levels_for(oracle), levels_for(activity)
    formula_vs_classifier = pd.DataFrame({"kind": pd.Series(ACCOUNT_KIND),
                                          "true harmful ratio": summary["true harmful ratio"],
                                          "level from true labels": base_true,
                                          "level from classifier": base_pred})
    print("\n(ii) Risk level from the true labels (formula only) and from the classifier:")
    display(formula_vs_classifier)
    print("true labels :", assess(base_true), "\nclassifier  :", assess(base_pred))

    # (iii) weights
    WEIGHT_SETS = {"base (0.5, 0.3, 0.2)": (0.5, 0.3, 0.2), "ratio-heavy (0.7, 0.2, 0.1)": (0.7, 0.2, 0.1),
                   "severity-heavy (0.3, 0.5, 0.2)": (0.3, 0.5, 0.2), "trend-heavy (0.4, 0.2, 0.4)": (0.4, 0.2, 0.4),
                   "equal (1/3 each)": (1 / 3, 1 / 3, 1 / 3)}
    rows = []
    for name, w in WEIGHT_SETS.items():
        for labels_used, frame, base in [("classifier", activity, base_pred), ("true labels", oracle, base_true)]:
            lv = levels_for(frame, w)
            rows.append({"weights": name, "labels": labels_used, **assess(lv),
                         "levels changed vs base": int((lv != base).sum()),
                         "levels": " ".join(f"{u[-1]}:{l[0]}" for u, l in lv.items())})
    weight_sensitivity = pd.DataFrame(rows)
    print("\n(iii) Weight sensitivity (cut-offs recomputed with the same reference-user rule):")
    with pd.option_context("display.max_colwidth", 60):
        display(weight_sensitivity)

    # (iv) cut-offs
    rows = []
    for medium_ratio in [0.10, 0.15, 0.20, 0.25, 0.30]:
        for high_ratio in [0.40, 0.50, 0.60]:
            lv = levels_for(activity, RISK_WEIGHTS, medium_ratio, high_ratio)
            rows.append({"Medium from r": medium_ratio, "High from r": high_ratio, **assess(lv),
                         "accounts rated High": int((lv == "High").sum())})
    cutoff_sensitivity = pd.DataFrame(rows)
    print("\n(iv) Cut-off sensitivity (classifier labels, base weights):")
    display(cutoff_sensitivity)

    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharex=True)
    for hr, g in cutoff_sensitivity.groupby("High from r"):
        axes[0].plot(g["Medium from r"], g["benign rated Medium/High"], "o-", label=f"High from r = {hr}")
        axes[1].plot(g["Medium from r"], g["risky rated Low"], "o-", label=f"High from r = {hr}")
    axes[0].set_title(f"Benign accounts rated Medium/High (of {len(BENIGN)})")
    axes[1].set_title(f"Risky accounts rated Low (of {len(RISKY)})")
    for ax in axes:
        ax.axvline(MEDIUM_RATIO, ls=":", color="gray"); ax.set_xlabel("Medium cut-off (harmful ratio)")
        ax.set_ylabel("Accounts"); ax.legend(fontsize=8)
    fig.tight_layout(); save_fig(fig, "7_risk_sensitivity"); plt.show()

    formula_vs_classifier.to_csv(os.path.join(RUN_DIR, "risk_formula_vs_classifier.csv"))
    weight_sensitivity.to_csv(os.path.join(RUN_DIR, "risk_weight_sensitivity.csv"), index=False)
    cutoff_sensitivity.to_csv(os.path.join(RUN_DIR, "risk_cutoff_sensitivity.csv"), index=False)
else:
    print("Real account data: the sensitivity analysis needs the simulated true labels and is skipped.")

---
# Section 9 · Ledger verification, tamper experiments, saved outputs, requirement checklist

**Tamper experiments (9.1).** Each scenario changes a **copy** of the ledger and then runs `verify()` and the
checkpoint check (3.1):

| # | Scenario | Expected |
|---|---|---|
| 1 | modify the content of an evidence block | `verify()` detects it (hash mismatch) |
| 2 | modify the recorded timestamp of a block | `verify()` detects it (hash mismatch) |
| 3 | delete an intermediate block | `verify()` detects it (broken link) |
| 4 | modify a block **and re-mine it and every later block** (coordinated rewrite) | `verify()` does **not** detect it; only the checkpoint does |
| 5 | delete an intermediate block and re-mine the rest | as 4 |
| 6 | cut off the newest blocks | `verify()` does **not** detect it; only the checkpoint does |

In [ ]:
# ============================================================
# 9.1  Integrity check + tamper experiments
# ============================================================
print("Blocks by type:", dict(ledger.count_by_type()))
ledger_ok = ledger.verify()
checkpoints_ok = ledger.verify_checkpoints()
print("Real ledger :", ledger_ok)
print("Checkpoints :", checkpoints_ok, [cp["label"] for cp in ledger.checkpoints])

def tampered_copy():
    t = copy.deepcopy(ledger)
    t.file_path = None            # a copy: never written to disk
    return t

evidence_ids = [b["index"] for b in ledger.chain if b["type"] == "EVIDENCE"]
target = evidence_ids[0] if evidence_ids else 1
middle = len(ledger.chain) // 2

def modify_content(t):
    data = t.chain[target]["data"]
    if "prediction" in data:
        data["prediction"] = "normal"     # someone tries to hide the evidence
    else:
        data["train_loss"] = 0.0

def modify_timestamp(t):
    t.chain[target]["timestamp"] = "2020-01-01T00:00:00+00:00"

def delete_middle(t):
    del t.chain[middle]

def rewrite_after_change(t):
    modify_content(t); t.remine_from(target)

def delete_and_rewrite(t):
    del t.chain[middle]; t.remine_from(middle)

def cut_off_newest(t):
    del t.chain[-3:]

SCENARIOS = [
    ("1 modify evidence content", f"block {target}: prediction -> normal", modify_content),
    ("2 modify timestamp", f"block {target}: timestamp -> 2020-01-01", modify_timestamp),
    ("3 delete intermediate block", f"block {middle} removed", delete_middle),
    ("4 modify + re-mine later blocks", f"block {target} changed, blocks {target}-{len(ledger.chain) - 1} re-mined",
     rewrite_after_change),
    ("5 delete + re-mine later blocks", f"block {middle} removed, later blocks re-mined", delete_and_rewrite),
    ("6 cut off newest blocks", "last 3 blocks removed", cut_off_newest),
]
rows = []
for name, what, change in SCENARIOS:
    t = tampered_copy()
    started = time.perf_counter()
    change(t)
    seconds = time.perf_counter() - started
    v, cp = t.verify(), t.verify_checkpoints(ledger.checkpoints)
    rows.append({"scenario": name, "change": what, "attack time (s)": round(seconds, 3),
                 "verify()": "detected" if not v[0] else "not detected", "verify() message": v[1],
                 "checkpoint": "detected" if not cp[0] else "not detected", "checkpoint message": cp[1],
                 "detected": (not v[0]) or (not cp[0])})
tamper_table = pd.DataFrame(rows)
with pd.option_context("display.max_colwidth", 70):
    display(tamper_table.drop(columns="detected"))
tamper_table.to_csv(os.path.join(RUN_DIR, "tamper_experiments.csv"), index=False)
tamper_result = (rows[0]["verify()"] != "detected", rows[0]["verify() message"])   # scenario 1, for the checklist
print("\nScenarios 4-6 pass verify(): a local hash chain alone cannot stop its holder from rewriting it.")
print("They are detected only because the head hash was recorded outside the ledger (checkpoint).")

In [ ]:
# ============================================================
# 9.2  Output files
# ============================================================
activity.drop(columns=["week"]).to_csv(os.path.join(RUN_DIR, "user_activity_analysed.csv"), index=False)
for root, _, files in os.walk(RUN_DIR):
    if "final_classifier" in root or "banglat5" in root:
        continue
    for name in sorted(files):
        print(os.path.join(root, name))
print("\nModels:", os.path.join(RUN_DIR, "final_classifier"),
      "| BanglaT5:", T5_DIR if TRAIN_BANGLAT5 else "not trained (QUICK mode)")
print(f"\nDataset version {DATA_VERSION} | seed {SEED} | run mode {RUN_MODE}")

In [ ]:
# ============================================================
# 9.3  Requirement checklist (filled in from the results of THIS run)
# ============================================================
def check(ok):
    return "✔" if ok else "✘"

ex1, ex2 = demo_results.iloc[0], demo_results.iloc[1]
checklist = [
    ("Requirement", "Public account + About info", "8.1",
     check(all(uid in USER_ABOUT for uid in profiles)), f"{len(USER_ABOUT)} accounts with About info"),
    ("Requirement", "Monitoring activities: status / share / comment", "8.1–8.2",
     check(set(activity["activity_type"]) >= {"status", "share", "comment"}),
     str(activity["activity_type"].value_counts().to_dict())),
    ("Requirement", "Activities as time-series data", "8.2, 8.4",
     check(activity["week"].nunique() > 1), f"{activity['week'].nunique()} weeks, weekly charts per user"),
    ("Requirement", "Profiling with 5 aspects -> profile vector", "8.3",
     check(all(len(p["profile_vector"]) == 7 for p in profiles.values())), "activity, harm ratio, type, trend, channel"),
    ("Requirement", "Result: activity profile + risk level", "8.3–8.5",
     check(len(summary) > 0), ", ".join(f"{u}: {l}" for u, l in summary["risk level"].items())),
    ("Novel", "kutt*r bacc*  ->  কুত্তার বাচ্চা", "2.5, 7.1",
     check(ex1["transliterated"] == "কুত্তার বাচ্চা"), ex1["transliterated"]),
    ("Novel", "কুত্*ার বাচ্চা  ->  কুত্তার বাচ্চা", "2.5, 7.1",
     check(ex2["transliterated"] == "কুত্তার বাচ্চা"), ex2["transliterated"]),
    ("Novel", "Both classified as harmful and recorded in the ledger", "7.1",
     check(pd.notna(ex1["block_index"]) and pd.notna(ex2["block_index"])),
     f"{ex1['prediction']} ({ex1['confidence']:.2f}), {ex2['prediction']} ({ex2['confidence']:.2f})"),
    ("Novel", "Problem 1: class imbalance reduced (training set)", "4.2",
     check(imbalance(after) < imbalance(before)),
     f"normal / smallest class: {imbalance(before):.1f}x -> {imbalance(after):.1f}x"),
    ("Novel", "Problem 2: macro F1 improved (test set)", "6.3",
     check(RUN_BASELINE and improved_result["macro_f1"] > baseline_result["macro_f1"]),
     f"{baseline_result['macro_f1']:.3f} -> {improved_result['macro_f1']:.3f}" if RUN_BASELINE else "baseline skipped"),
    ("Novel", "Problem 2: F1 improved for the harmful classes", "6.3",
     check(RUN_BASELINE and all(table.loc[l, "change"] > 0 for l in LABELS[1:])),
     ", ".join(f"{l}: {table.loc[l, 'change']:+.2f}" for l in LABELS) if RUN_BASELINE else "baseline skipped"),
    ("Revision", "Normalisation evaluated on held-out items", "2.6",
     check(len(norm_summary) > 0),
     (f"exact match {norm_summary.loc['ALL'].filter(like='exact match').iloc[-1]:.2f}, word accuracy "
      f"{norm_summary.loc['ALL'].filter(like='word accuracy').iloc[-1]:.2f}") if len(norm_summary) else "skipped"),
    ("Shared", "Simulated federated learning (clients never share data)", "5",
     check(ledger.count_by_type()["CLIENT_UPDATE"] > 0),
     f"{NUM_CLIENTS} clients, {FL_ROUNDS} rounds, {ledger.count_by_type()['CLIENT_UPDATE']} client updates"),
    ("Shared", "Ledger valid + every tamper scenario detected (verify or checkpoint)", "9.1",
     check(ledger_ok[0] and checkpoints_ok[0] and tamper_table["detected"].all()),
     f"{int(tamper_table['detected'].sum())} of {len(tamper_table)} scenarios detected; "
     f"{int((tamper_table['verify()'] == 'not detected').sum())} only by the checkpoint"),
]
checklist = pd.DataFrame(checklist, columns=["part", "requirement", "cell", "ok", "result in this run"])
if SIMULATED_EXPECTATION:
    extra = pd.DataFrame([("Requirement", f"Simulated {u} shows: {c} = {v}", "8.3",
                           check(summary.loc[u, "found?"] == "yes"), str(summary.loc[u, c]))
                          for u, (c, v) in SIMULATED_EXPECTATION.items()], columns=checklist.columns)
    checklist = pd.concat([checklist, extra], ignore_index=True)
checklist.to_csv(os.path.join(RUN_DIR, "requirement_checklist.csv"), index=False)
with pd.option_context("display.max_colwidth", 90):
    display(checklist)
if RUN_MODE == "QUICK":
    print("QUICK mode: model scores are lower than in FULL mode; the reported results come from FULL mode.")

---
# Section 10 · Robustness experiments (thesis revision)

| Cell | Question | Method |
|---|---|---|
| 10.1 | runs the extra configurations (already saved runs are skipped, so the cell can be restarted) | seeds 43, 44 (IID); for version C also centralized, Dirichlet and source-based Non-IID for seeds 42–44 |
| 10.2 | Are the results stable across seeds? Is the improved model better than the baseline in every seed? | mean ± std of accuracy, macro precision, recall and F1; per-class F1; paired bootstrap per seed |
| 10.3 | What does federated training cost compared with centralized training, and how does client heterogeneity affect it? | centralized vs IID vs Non-IID: macro and per-class F1, convergence |
| 10.4 | How reliable are the labels, and how does the model score against verified reference labels? | two independent human annotators, Cohen's kappa, adjudication, re-evaluation |

| 10.5 | thesis figures comparing the dataset versions (seed 42) | per-class F1 and training curves per version |

Run the notebook once per dataset version, in the order **A, B, C**: every version writes its own folder
`version_<V>/`, all runs are collected in the shared `runs_v2/`, and the run of version C (last) adds the
federated-learning study and produces the summaries of all versions. The extra runs use their own ledger file
(`experiments_ledger_<V>.json`), so the main ledger of Sections 3–9 is not changed.

In [ ]:
# ============================================================
# 10.1  Extra seeds and the federated-learning study (results in 10.2 and 10.3)
# ============================================================
experiment_ledger = HashLinkedLedger(os.path.join(RUN_DIR, f"experiments_ledger_{DATA_VERSION}.json"))

def free_gpu():
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

def run_file(seed, training, partition):
    return os.path.join(RUNS_DIR, f"{DATA_VERSION}_{training}_{partition}_seed{seed}_{RUN_MODE}.json")

def run_configuration(seed, training, partition):
    """Baseline + improved model, exactly as in 6.1-6.2, for one seed / training mode / client split."""
    tag = f"{training}-{partition}-seed{seed}"
    clients = None
    if training == "federated":
        clients = split_into_clients(train_df, NUM_CLIENTS, partition, seed)
        balanced = [balance_training_data(c, seed) for c in clients]
        for i, c in enumerate(clients, 1):
            print(f"Client {i}: {c['label'].value_counts().reindex(LABELS, fill_value=0).to_dict()}")
        train_baseline = lambda: run_federated(clients, False, f"baseline [{tag}]", experiment_ledger)
        train_improved = lambda: run_federated(balanced, USE_CLASS_WEIGHTS, f"improved [{tag}]", experiment_ledger)
    else:                     # centralized: the same over-sampling, applied to all training data at once
        balanced_all = balance_training_data(train_df, seed)
        train_baseline = lambda: run_centralized(train_df, f"baseline [{tag}]", experiment_ledger)
        train_improved = lambda: run_centralized(balanced_all, f"improved [{tag}]", experiment_ledger)

    set_seed(seed)
    m, history_base = train_baseline()
    base_res = evaluate(m, test_df)
    del m; free_gpu()

    set_seed(seed)
    m, history_impr = train_improved()
    over_res = evaluate(m, test_df)
    bias, _ = tune_class_bias(predict_proba(m, list(val_df["clean_text"])), [LABEL2ID[l] for l in val_df["label"]])
    impr_res = evaluate(m, test_df, bias=bias)
    del m; free_gpu()
    print(f"[{tag}] test macro F1: baseline {base_res['macro_f1']:.4f} -> improved {impr_res['macro_f1']:.4f}")
    return save_run(seed, training, partition, base_res, over_res, impr_res, bias,
                    {"baseline": history_base, "improved": history_impr}, clients)

configs = []
if RUN_EXTRA_SEEDS:
    configs += [(s, "federated", "iid") for s in EXTRA_SEEDS]
if RUN_FL_STUDY and DATA_VERSION == "C":
    for s in [SEED] + EXTRA_SEEDS:
        configs += [(s, "centralized", "all")] + [(s, "federated", p) for p in FL_STUDY_PARTITIONS]
print(f"{len(configs)} configurations for dataset version {DATA_VERSION}")

for seed, training, partition in configs:
    path = run_file(seed, training, partition)
    if os.path.isfile(path):
        print("already saved:", os.path.basename(path))
        continue
    started = time.time()
    run_configuration(seed, training, partition)
    print(f"saved {os.path.basename(path)} ({(time.time() - started) / 60:.1f} min)")

set_seed(SEED)          # the rest of the notebook continues with the main seed
print("Experiment ledger:", experiment_ledger.verify())

In [ ]:
# ============================================================
# 10.2  Mean ± std over seeds, per-class F1, consistency and paired bootstrap
# ============================================================
METRICS = {"accuracy": "Accuracy", "macro_precision": "Precision (macro)", "macro_recall": "Recall (macro)",
           "macro_f1": "Macro F1"}

def load_runs():
    runs = [json.load(open(p, encoding="utf-8")) for p in sorted(glob.glob(os.path.join(RUNS_DIR, "*.json")))]
    return [r for r in runs if r["run_mode"] == RUN_MODE]           # never mix QUICK and FULL numbers

def mean_std(values, digits=4):
    values = [float(v) for v in values if v is not None and not pd.isna(v)]
    if not values:
        return ""
    if len(values) == 1:
        return f"{values[0]:.{digits}f} (1 seed)"
    return f"{np.mean(values):.{digits}f} ± {np.std(values, ddof=1):.{digits}f}"

def macro_f1_of(y_true, y_pred):
    return f1_score(y_true, y_pred, labels=range(len(LABELS)), average="macro", zero_division=0)

def paired_bootstrap(y_true, pred_a, pred_b, n=1000, seed=0):
    """Resample the test comments with replacement (the same rows for both models) and return the 95% interval
    of macro-F1(b) - macro-F1(a) and the share of resamples in which b is better."""
    y_true, pred_a, pred_b = map(np.asarray, (y_true, pred_a, pred_b))
    rng = np.random.default_rng(seed)
    diffs = np.empty(n)
    for i in range(n):
        idx = rng.integers(0, len(y_true), len(y_true))
        diffs[i] = macro_f1_of(y_true[idx], pred_b[idx]) - macro_f1_of(y_true[idx], pred_a[idx])
    low, high = np.percentile(diffs, [2.5, 97.5])
    return float(low), float(high), float((diffs > 0).mean())

runs = load_runs()
iid_runs = [r for r in runs if r["training"] == "federated" and r["partition"] == "iid"]
records = [{"version": r["data_version"], "seed": r["seed"], "model": m, **{k: r[m][k] for k in METRICS},
            **{l: r[m]["per_class_f1"][l] for l in LABELS},
            "checked": r["checked"].get("macro_f1") if (m == "improved" and r.get("checked")) else None}
           for r in iid_runs for m in ("baseline", "improved") if r.get(m)]
seed_runs = pd.DataFrame(records)

if len(seed_runs):
    # (a) mean ± std over seeds
    seed_summary = pd.DataFrame([
        {"version": v, "model": m, "seeds": ", ".join(map(str, sorted(g["seed"]))),
         **{name: mean_std(g[k]) for k, name in METRICS.items()},
         "Macro F1 (checked labels)": mean_std(g["checked"]) if m == "improved" else ""}
        for (v, m), g in seed_runs.groupby(["version", "model"])])
    seed_per_class = pd.DataFrame([
        {"version": v, "model": m, **{l: mean_std(g[l]) for l in LABELS}}
        for (v, m), g in seed_runs.groupby(["version", "model"])])
    print("(a) Test set, mean ± standard deviation over seeds (federated, IID):")
    display(seed_summary); display(seed_per_class)

    # (b) improved vs baseline in every seed + paired bootstrap on the test set
    rows = []
    for r in iid_runs:
        if not r.get("baseline"):
            continue
        low, high, p_better = paired_bootstrap(r["test_true"], r["test_pred"]["baseline"], r["test_pred"]["improved"])
        rows.append({"version": r["data_version"], "seed": r["seed"],
                     "baseline macro F1": r["baseline"]["macro_f1"], "improved macro F1": r["improved"]["macro_f1"],
                     "difference": round(r["improved"]["macro_f1"] - r["baseline"]["macro_f1"], 4),
                     "classes improved": sum(r["improved"]["per_class_f1"][l] > r["baseline"]["per_class_f1"][l]
                                             for l in LABELS),
                     "bootstrap 95% CI": f"[{low:+.4f}, {high:+.4f}]",
                     "CI": "above 0" if low > 0 else ("below 0" if high < 0 else "includes 0"),
                     "P(improved better)": round(p_better, 3)})
    consistency = pd.DataFrame(rows).sort_values(["version", "seed"])
    print("(b) Improved minus baseline, every seed:")
    display(consistency)
    for v, g in consistency.groupby("version"):
        print(f"Version {v}: difference {mean_std(g['difference'])} over {len(g)} seeds | improved better in "
              f"{int((g['difference'] > 0).sum())} of {len(g)} seeds | CI above 0: {int((g['CI'] == 'above 0').sum())}, "
              f"below 0: {int((g['CI'] == 'below 0').sum())}, includes 0: {int((g['CI'] == 'includes 0').sum())}")
    print("Three seeds are too few for a significance test ACROSS seeds; none is claimed. The bootstrap interval "
          "describes the uncertainty caused by the finite test set within one seed.")

    # (c) chart: per-class F1, mean ± std, baseline vs improved, per version
    versions = sorted(seed_runs["version"].unique())
    fig, axes = plt.subplots(1, len(versions), figsize=(5.5 * len(versions), 3.8), squeeze=False)
    for ax, v in zip(axes[0], versions):
        x = np.arange(len(LABELS) + 1)
        for j, (m, color) in enumerate([("baseline", "#95A5A6"), ("improved", "#27AE60")]):
            g = seed_runs[(seed_runs["version"] == v) & (seed_runs["model"] == m)]
            if not len(g):
                continue
            means = [g[l].mean() for l in LABELS] + [g["macro_f1"].mean()]
            stds = [g[l].std(ddof=1) for l in LABELS] + [g["macro_f1"].std(ddof=1)]
            ax.bar(x + (j - 0.5) * 0.4, means, 0.4, yerr=np.nan_to_num(stds), capsize=3, label=m, color=color)
        ax.set_xticks(x); ax.set_xticklabels(LABELS + ["macro F1"], rotation=30)
        ax.set_ylim(0, 1); ax.set_title(f"Version {v} (mean ± std over seeds)"); ax.legend()
    fig.tight_layout(); save_fig(fig, "8_seeds_per_class_f1"); plt.show()

    seed_runs.to_csv(os.path.join(RUN_DIR, "seed_runs.csv"), index=False)
    seed_summary.to_csv(os.path.join(RUN_DIR, "seed_summary.csv"), index=False)
    seed_per_class.to_csv(os.path.join(RUN_DIR, "seed_per_class_f1.csv"), index=False)
    consistency.to_csv(os.path.join(RUN_DIR, "seed_consistency_bootstrap.csv"), index=False)
else:
    print("No saved runs yet.")

In [ ]:
# ============================================================
# 10.3  Centralized vs IID vs Non-IID federated learning (dataset version C)
# ============================================================
def setting_of(r):
    return "centralized" if r["training"] == "centralized" else f"federated {r['partition']}"

def heterogeneity(r):
    """Mean total-variation distance between each client's class mix and the overall class mix (0 = IID)."""
    counts = np.array(r.get("client_label_counts") or [], float)
    if counts.size == 0:
        return np.nan
    client_mix = counts / counts.sum(axis=1, keepdims=True)
    overall = counts.sum(axis=0) / counts.sum()
    return float(0.5 * np.abs(client_mix - overall).sum(axis=1).mean())

def best_round(r, model):
    h = r["history"][model]
    return h["round"][int(np.argmax(h["macro_f1"]))]

def rounds_to_95(r, model):
    h = r["history"][model]
    target = 0.95 * max(h["macro_f1"])
    return next(rd for rd, f in zip(h["round"], h["macro_f1"]) if f >= target)

SETTINGS = ["centralized", "federated iid", "federated dirichlet", "federated source"]
study = [r for r in load_runs() if r["data_version"] == "C" and r.get("baseline")]
rows, per_class_rows = [], []
for s in SETTINGS:
    rs = [r for r in study if setting_of(r) == s]
    if not rs:
        continue
    rows.append({"setting": s, "seeds": len(rs),
                 "client heterogeneity": mean_std([heterogeneity(r) for r in rs], 3) if s != "centralized" else "-",
                 "baseline macro F1": mean_std([r["baseline"]["macro_f1"] for r in rs]),
                 "improved macro F1": mean_std([r["improved"]["macro_f1"] for r in rs]),
                 "improved accuracy": mean_std([r["improved"]["accuracy"] for r in rs]),
                 "best round (improved)": mean_std([best_round(r, "improved") for r in rs], 1),
                 "rounds to 95% of best": mean_std([rounds_to_95(r, "improved") for r in rs], 1)})
    per_class_rows.append({"setting": s, **{l: mean_std([r["improved"]["per_class_f1"][l] for r in rs])
                                            for l in LABELS}})

if rows:
    fl_study = pd.DataFrame(rows)
    fl_study_per_class = pd.DataFrame(per_class_rows)
    print("Test set, mean ± std over seeds (version C). Rounds = epochs for centralized training.")
    display(fl_study); display(fl_study_per_class)
    fl_study.to_csv(os.path.join(RUN_DIR, "fl_study.csv"), index=False)
    fl_study_per_class.to_csv(os.path.join(RUN_DIR, "fl_study_per_class_f1.csv"), index=False)

    # Convergence: validation macro F1 per round, mean over seeds (improved model)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for s, color in zip(SETTINGS, ["#2C3E50", "#27AE60", "#E67E22", "#8E44AD"]):
        rs = [r for r in study if setting_of(r) == s]
        if not rs:
            continue
        curves = np.array([r["history"]["improved"]["macro_f1"] for r in rs])
        rounds = rs[0]["history"]["improved"]["round"]
        axes[0].plot(rounds, curves.mean(axis=0), "o-", color=color, label=s)
        axes[0].fill_between(rounds, curves.min(axis=0), curves.max(axis=0), color=color, alpha=0.15)
    axes[0].set_xlabel("Round (epoch for centralized)"); axes[0].set_ylabel("Validation macro F1")
    axes[0].set_title("Convergence (improved model, mean and range over seeds)"); axes[0].legend()

    # Class mix of the clients (seed SEED) for each federated split
    mixes = []
    for s in SETTINGS[1:]:
        r = next((r for r in study if setting_of(r) == s and r["seed"] == SEED), None)
        if r and r.get("client_label_counts"):
            for k, c in enumerate(r["client_label_counts"], 1):
                mixes.append(pd.Series(np.array(c) / sum(c), index=LABELS, name=f"{s.split()[-1]} {k}"))
    if mixes:
        pd.DataFrame(mixes).plot(kind="bar", stacked=True, ax=axes[1], width=0.8,
                                 color=[CLASS_COLORS[c] for c in LABELS])
        axes[1].set_ylabel("Share of the client's comments"); axes[1].set_title("Class mix per client")
        axes[1].legend(fontsize=8, ncol=5, loc="lower center"); axes[1].tick_params(axis="x", rotation=30)
    fig.tight_layout(); save_fig(fig, "9_fl_study"); plt.show()
else:
    print("No runs of the federated-learning study yet (run 10.1 with DATA_VERSION = 'C').")

In [ ]:
# ============================================================
# 10.4  Human reference labels: inter-annotator agreement and re-evaluation
# ============================================================
# 400 test comments (80 per class, spread over the sources) were labelled independently by two annotators,
# blind to the source label, with dataset/guideline_v1.md. Disagreements were resolved by adjudication.
# Tool and protocol: dataset/annotation/annotation_tool.py. The file holds IDs and labels only.
ANNOTATION_FILE = os.path.join("dataset", "annotation", "annotations.csv")
LOCAL_ANNOTATIONS = "annotations.csv"

def load_annotations():
    for path in (ANNOTATION_FILE, LOCAL_ANNOTATIONS):
        if os.path.isfile(path):
            return pd.read_csv(path)
    try:
        download_from_github("dataset/annotation/annotations.csv", LOCAL_ANNOTATIONS)
        return pd.read_csv(LOCAL_ANNOTATIONS)
    except Exception as e:
        print(f"No annotation file yet ({type(e).__name__}). Run dataset/annotation/annotation_tool.py first.")
        return None

def landis_koch(k):
    return ("poor" if k < 0 else "slight" if k <= 0.2 else "fair" if k <= 0.4 else
            "moderate" if k <= 0.6 else "substantial" if k <= 0.8 else "almost perfect")

ann = load_annotations()
if ann is not None:
    both = ann[ann["annotator_1"].isin(LABELS) & ann["annotator_2"].isin(LABELS)]
    kappa = cohen_kappa_score(both["annotator_1"], both["annotator_2"], labels=LABELS)
    print(f"Annotated comments: {len(ann)} | labelled by both (not 'unclear'): {len(both)}")
    print(f"Raw agreement {np.mean(both['annotator_1'] == both['annotator_2']):.1%} | "
          f"Cohen's kappa {kappa:.3f} ({landis_koch(kappa)} agreement, Landis & Koch)")

    pairs = {"annotator 1 vs annotator 2": ("annotator_1", "annotator_2"),
             "annotator 1 vs source label": ("annotator_1", "source_label"),
             "annotator 2 vs source label": ("annotator_2", "source_label"),
             "adjudicated vs source label": ("adjudicated", "source_label")}
    agreement = []
    for name, (a, b) in pairs.items():
        rows = ann[ann[a].isin(LABELS) & ann[b].isin(LABELS)]
        k = cohen_kappa_score(rows[a], rows[b], labels=LABELS)
        agreement.append({"comparison": name, "rows": len(rows), "raw agreement": round(float((rows[a] == rows[b]).mean()), 3),
                          "Cohen's kappa": round(float(k), 3), "interpretation": landis_koch(k)})
    agreement = pd.DataFrame(agreement)
    per_class_kappa = pd.Series({l: round(float(cohen_kappa_score(both["annotator_1"] == l, both["annotator_2"] == l)), 3)
                                 for l in LABELS}, name="kappa (one class vs rest)")
    display(agreement); display(per_class_kappa.to_frame())

    disagreement_matrix = pd.crosstab(both["annotator_1"], both["annotator_2"]).reindex(index=LABELS, columns=LABELS,
                                                                                         fill_value=0)
    disagreement_by_class = (ann.assign(disagree=ann["annotator_1"] != ann["annotator_2"])
                             .groupby("source_label")["disagree"].agg(["sum", "mean"]).reindex(LABELS)
                             .rename(columns={"sum": "disagreements", "mean": "share"}).round(3))
    print("Annotator 1 (rows) vs annotator 2 (columns):"); display(disagreement_matrix)
    print("Disagreements per source-label class:"); display(disagreement_by_class)

    # Re-evaluate the improved model on the adjudicated reference labels
    ref = ann[ann["adjudicated"].isin(LABELS)].set_index("id")["adjudicated"]
    ids = test_df["id"].astype(str).to_numpy()
    in_ref = np.isin(ids, ref.index.astype(str))
    y_ref = ref.reindex(ids[in_ref]).map(LABEL2ID).to_numpy()
    y_src = np.asarray(improved_result["y_true"])[in_ref]

    def against(y_true, y_pred):
        per_class = f1_score(y_true, y_pred, labels=range(len(LABELS)), average=None, zero_division=0)
        return {"accuracy": accuracy_score(y_true, y_pred), "macro F1": macro_f1_of(y_true, y_pred),
                **dict(zip(LABELS, per_class))}

    rows = []
    for r in [r for r in load_runs() if r["data_version"] == DATA_VERSION and r["training"] == "federated"
              and r["partition"] == "iid"]:
        pred = pd.Series(r["test_pred"]["improved"], index=r["test_ids"]).reindex(ids[in_ref]).to_numpy()
        rows.append({"seed": r["seed"], "labels": "source labels", **against(y_src, pred)})
        rows.append({"seed": r["seed"], "labels": "human reference labels", **against(y_ref, pred)})
    reference_eval = pd.DataFrame(rows)
    reference_summary = pd.DataFrame([{"labels": lab, "seeds": len(g),
                                       **{c: mean_std(g[c]) for c in ["accuracy", "macro F1", *LABELS]}}
                                      for lab, g in reference_eval.groupby("labels", sort=False)])
    print(f"Improved model on the {int(in_ref.sum())} annotated test comments (mean ± std over seeds):")
    display(reference_summary)

    agreement.to_csv(os.path.join(RUN_DIR, "annotation_agreement.csv"), index=False)
    per_class_kappa.to_csv(os.path.join(RUN_DIR, "annotation_per_class_kappa.csv"))
    disagreement_matrix.to_csv(os.path.join(RUN_DIR, "annotation_disagreement_matrix.csv"))
    disagreement_by_class.to_csv(os.path.join(RUN_DIR, "annotation_disagreement_by_class.csv"))
    reference_summary.to_csv(os.path.join(RUN_DIR, "reference_label_evaluation.csv"), index=False)

In [ ]:
# ============================================================
# 10.5  Figures comparing the dataset versions (seed SEED, federated IID)
# ============================================================
version_runs = {r["data_version"]: r for r in load_runs()
                if r["training"] == "federated" and r["partition"] == "iid" and r["seed"] == SEED}
if len(version_runs) > 1:
    versions = sorted(version_runs)
    fig, ax = plt.subplots(figsize=(10, 4))
    x = np.arange(len(LABELS) + 1)
    width = 0.8 / len(versions)
    for j, v in enumerate(versions):
        res = version_runs[v]["improved"]
        bars = ax.bar(x + (j - (len(versions) - 1) / 2) * width,
                      [res["per_class_f1"][l] for l in LABELS] + [res["macro_f1"]], width, label=f"Version {v}")
        ax.bar_label(bars, fmt="%.2f", fontsize=7)
    ax.set_xticks(x); ax.set_xticklabels(LABELS + ["macro F1"]); ax.set_ylim(0, 1)
    ax.set_ylabel("F1 (test set)"); ax.set_title(f"Improved model per dataset version (seed {SEED})"); ax.legend()
    save_fig(fig, "10_versions_per_class_f1"); plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
    for ax, model_name in zip(axes, ["baseline", "improved"]):
        for v in versions:
            h = version_runs[v]["history"].get(model_name)
            if h:
                line, = ax.plot(h["round"], h["macro_f1"], "o-", label=f"Version {v}")
                best = int(np.argmax(h["macro_f1"]))
                ax.plot(h["round"][best], h["macro_f1"][best], "*", ms=14, color=line.get_color())
        ax.set_title(f"{model_name.capitalize()} run"); ax.set_xlabel("Federated round"); ax.legend()
    axes[0].set_ylabel("Validation macro F1")
    fig.tight_layout(); save_fig(fig, "11_versions_rounds"); plt.show()
else:
    print("Run the notebook for more than one dataset version first (A, B, then C).")

---
# Summary

| Part | What | Method | Where |
|---|---|---|---|
| **Requirement** | DF analysis of a public Facebook account | activities (status / share / comment) as weekly time series, About info, 5-aspect profile, risk level, sensitivity analysis | 8 |
| **Novel** | Unmask hidden words | pattern search: bad phrases (with the neighbouring word) first, then single words | 2.2 |
| **Novel** | Transliteration | word dictionary learned from BanTH + spelling keys + BanglaT5 for unknown words | 2.3, 2.4 |
| **Novel** | Normalisation evaluation | 300 held-out items (masked, Banglish, mixed), exact match + word accuracy, failure analysis | 2.6 |
| **Novel** | Harmful comment → ledger evidence | clean → unmask → transliterate → classify → record if harmful | 3, 7 |
| **Novel** | Forensic dataset | 7 public sources combined with one label guideline, duplicates and conflicting labels removed, probable label errors removed from training (confident learning), fixed split | 1, `dataset/` |
| **Novel** | Class imbalance (Problem 1) | balanced training core (version C) + over-sampling the rare harmful classes inside every client | 1, 4 |
| **Novel** | Per-class F1 (Problem 2) | per-class decision thresholds tuned on the validation set; baseline vs improved over three seeds | 6, 10.2 |
| Shared | Classifier | BanglaBERT, 3 simulated federated clients, FedAvg, best round by validation macro F1; centralized and Non-IID comparison | 5, 10.3 |
| Shared | Evidence integrity | SHA-256 hash-linked ledger with proof-of-work and external checkpoints; six tamper scenarios | 3, 9 |
| Shared | Label reliability | two independent human annotators on 400 test comments, Cohen's kappa, adjudication | 10.4 |

## Limitations

* The monitored accounts in Section 8 are **simulated** from real test comments; no real account data was analysed.
  The profiling results are simulation-based and need validation on lawfully collected, labelled real accounts.
* Federated learning is **simulated on one GPU** with three clients. Sharing model weights is not a formal privacy
  guarantee (gradient inversion, membership inference); differential privacy and secure aggregation are not implemented.
* The ledger is a **single local file** with a low proof-of-work difficulty. It detects a change to any stored block, but
  a coordinated rewrite by its holder is detected only through an independent checkpoint (9.1). It provides
  tamper-evident records and chain-of-custody documentation, not legal admissibility on its own.
* **Label noise** between the sources limits per-class F1 more than the class imbalance does (6.5, 10.4).
* The unmasker can only restore words that occur in its vocabulary (2.6).
* The risk-score weights and cut-offs are set by design and tested by sensitivity analysis (8.6), not calibrated on
  labelled real accounts.